In [1]:
# Stage 2: multidegree extraction and phantom interactions

%load_ext autoreload
%autoreload 2

from pathlib import Path
from itertools import product, combinations, permutations

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import common_function as cf


# ------------------------------------------------------------------
# Reproducibility and output directories
# ------------------------------------------------------------------

seed = 20260805
rng = np.random.default_rng(seed)

tol = 1e-10

result_dir = Path("results_stage2")
figure_dir = Path("figures_stage2")

result_dir.mkdir(exist_ok=True)
figure_dir.mkdir(exist_ok=True)


np.set_printoptions(
    precision=6,
    suppress=True
)


# ------------------------------------------------------------------
# Confirm the imported module
# ------------------------------------------------------------------

print(
    "Loaded common_function from:"
)

print(
    Path(cf.__file__).resolve()
)

print()

print(
    "Convention version:"
)

print(
    cf.CONVENTION_VERSION
)

print()


# ------------------------------------------------------------------
# Required Stage 2 infrastructure
# ------------------------------------------------------------------

required_functions = [
    "protocol_unitary",
    "protocol_log",
    "protocol_log_eig",
    "unitary_branch_distance",
    "multidegree_component",
    "converged_component",
    "richardson_component",
    "weight_decomposition",
    "pair_ledger",
    "palindrome_log",
    "group_commutator_unitary",
    "run_convention_selftests"
]


missing_functions = [
    name
    for name in required_functions
    if not hasattr(cf, name)
]


assert not missing_functions, (
    "Missing common functions: "
    + ", ".join(missing_functions)
)


print(
    "Required Stage 2 functions: PASS"
)

print()


# ------------------------------------------------------------------
# Frozen convention gate
# ------------------------------------------------------------------

cf.run_convention_selftests(
    verbose=True
)

Loaded common_function from:
C:\Users\liu.xuanc\Desktop\Code\Quantum-HOC\common_function.py

Convention version:
frozen-v1 (chronological: first-acting rightmost; L anti-Hermitian; K = iL)

Required Stage 2 functions: PASS

  [PASS] even (1,1) = +1/2 [H1,H2]: 5.18e-11
  [PASS] odd (1,1,1) = i(C1/3 - C2/6): 2.91e-05
  [PASS] odd reversal-invariant: 7.68e-11
  [PASS] even reversal flip: 6.27e-12
  [PASS] (2,1) = (i/12)[H1,[H1,H2]]: 2.37e-04
  [PASS] palindrome (1,1,1) = i(C1/12 - C2/6): 1.13e-04
  [PASS] witness: ||W-target|| << ||W-1||: 1.63e-01
  [PASS] weight_decomposition sanity: 0.00e+00


True

In [3]:
# ------------------------------------------------------------------
# Fixed generic reference triangle
# ------------------------------------------------------------------

N = 3

A, B, C = 0, 1, 2

node_labels = {
    A: "A",
    B: "B",
    C: "C"
}


# Use a dedicated RNG so that this reference system does not change
# when unrelated random experiments are inserted elsewhere.

reference_rng = np.random.default_rng(
    20260805
)


J_AB = reference_rng.uniform(
    -1.0,
    1.0,
    size=(3, 3)
)

J_BC = reference_rng.uniform(
    -1.0,
    1.0,
    size=(3, 3)
)

J_AC = reference_rng.uniform(
    -1.0,
    1.0,
    size=(3, 3)
)


H_AB = cf.general_edge_hamiltonian(
    A, B,
    J_AB,
    N
)

H_BC = cf.general_edge_hamiltonian(
    B, C,
    J_BC,
    N
)

H_AC = cf.general_edge_hamiltonian(
    A, C,
    J_AC,
    N
)


# Frozen chronological naming

H1 = H_AB
H2 = H_BC
H3 = H_AC


reference_edges = {
    "AB": H_AB,
    "BC": H_BC,
    "AC": H_AC
}


reference_couplings = {
    "AB": J_AB,
    "BC": J_BC,
    "AC": J_AC
}

In [4]:
# ------------------------------------------------------------------
# Basic operator diagnostics
# ------------------------------------------------------------------

edge_diagnostics = []


for edge_name, H_edge in reference_edges.items():

    edge_diagnostics.append({
        "edge": edge_name,

        "hermiticity_error":
            np.linalg.norm(
                H_edge - H_edge.conj().T
            ),

        "frobenius_norm":
            np.linalg.norm(H_edge)
    })


edge_diagnostics = pd.DataFrame(
    edge_diagnostics
)

display(edge_diagnostics)


pairwise_commutator_diagnostics = pd.DataFrame({

    "commutator": [
        "[H_AB, H_BC]",
        "[H_AB, H_AC]",
        "[H_BC, H_AC]"
    ],

    "norm": [
        np.linalg.norm(
            cf.commutator(H_AB, H_BC)
        ),

        np.linalg.norm(
            cf.commutator(H_AB, H_AC)
        ),

        np.linalg.norm(
            cf.commutator(H_BC, H_AC)
        )
    ]
})


display(
    pairwise_commutator_diagnostics
)

,edge,hermiticity_error,frobenius_norm
0,AB,0.0,4.994867
1,BC,0.0,4.920132
2,AC,0.0,5.681497


,commutator,norm
0,"[H_AB, H_BC]",16.120084
1,"[H_AB, H_AC]",17.501407
2,"[H_BC, H_AC]",15.994293


In [5]:
# ------------------------------------------------------------------
# Two-dimensional intrinsic cycle sector
# ------------------------------------------------------------------

C_AB, C_BC = cf.channel_basis(
    H_AB,
    H_BC,
    H_AC
)


C_AC = 0.25 * cf.commutator(
    H_AC,
    cf.commutator(
        H_AB,
        H_BC
    )
)


jacobi_residual = np.linalg.norm(
    C_AC - C_AB - C_BC
)


cycle_basis_matrix = np.column_stack([
    C_AB.reshape(-1),
    C_BC.reshape(-1)
])


cycle_basis_rank = np.linalg.matrix_rank(
    cycle_basis_matrix,
    tol=tol
)


cycle_diagnostics = pd.DataFrame({

    "quantity": [
        "||C_AB||",
        "||C_BC||",
        "||C_AC||",
        "||C_AC - C_AB - C_BC||",
        "cycle basis rank"
    ],

    "value": [
        np.linalg.norm(C_AB),
        np.linalg.norm(C_BC),
        np.linalg.norm(C_AC),
        jacobi_residual,
        cycle_basis_rank
    ]
})


display(
    cycle_diagnostics
)

,quantity,value
0,||C_AB||,6.572876e+00
1,||C_BC||,8.869948e+00
2,||C_AC||,1.024495e+01
3,||C_AC - C_AB - C_BC||,2.700006e-15
4,cycle basis rank,2.000000e+00


In [6]:
np.savez(
    result_dir / "reference_triangle_couplings.npz",

    J_AB=J_AB,
    J_BC=J_BC,
    J_AC=J_AC,

    seed=20260805
)


edge_diagnostics.to_csv(
    result_dir / "reference_edge_diagnostics.csv",
    index=False
)


pairwise_commutator_diagnostics.to_csv(
    result_dir / "reference_pairwise_commutators.csv",
    index=False
)

Multidegree extractor

In [7]:
# ------------------------------------------------------------------
# Analytic multidegree anchors
# ------------------------------------------------------------------

Cmt = cf.commutator


multidegree_targets = {

    "L_110": (
        0.5
        * Cmt(H1, H2)
    ),

    "L_210": (
        1j / 12
        * Cmt(
            H1,
            Cmt(H1, H2)
        )
    ),

    "L_120": (
        -1j / 12
        * Cmt(
            H2,
            Cmt(H1, H2)
        )
    ),

    "L_111": (
        1j
        * (
            (1 / 3)
            * Cmt(
                H1,
                Cmt(H2, H3)
            )

            - (1 / 6)
            * Cmt(
                H2,
                Cmt(H1, H3)
            )
        )
    )
}


component_specs = {

    "L_110": {
        "degrees": (1, 1, 0),
        "step": 0.004
    },

    "L_210": {
        "degrees": (2, 1, 0),
        "step": 0.02
    },

    "L_120": {
        "degrees": (1, 2, 0),
        "step": 0.02
    },

    "L_111": {
        "degrees": (1, 1, 1),
        "step": 0.01
    }
}


def relative_operator_error(
    observed,
    target
):

    return (
        np.linalg.norm(
            observed - target
        )
        / max(
            np.linalg.norm(target),
            1e-300
        )
    )


anchor_rows = []

reference_hamiltonians = [
    H1,
    H2,
    H3
]


for component_name, specification in component_specs.items():

    degrees = specification["degrees"]
    step = specification["step"]

    target = multidegree_targets[
        component_name
    ]


    (
        richardson,
        coarse,
        fine,
        coarse_fine_deviation
    ) = cf.richardson_component(
        reference_hamiltonians,
        degrees,
        step
    )


    coarse_error = relative_operator_error(
        coarse,
        target
    )

    fine_error = relative_operator_error(
        fine,
        target
    )

    richardson_error = relative_operator_error(
        richardson,
        target
    )


    antihermiticity_error = (
        np.linalg.norm(
            richardson
            + richardson.conj().T
        )
        / max(
            np.linalg.norm(richardson),
            1e-300
        )
    )


    anchor_rows.append({

        "component": component_name,

        "degrees": str(degrees),

        "step": step,

        "target_norm":
            np.linalg.norm(target),

        "coarse_error":
            coarse_error,

        "fine_error":
            fine_error,

        "richardson_error":
            richardson_error,

        "coarse_fine_deviation":
            coarse_fine_deviation,

        "antihermiticity_error":
            antihermiticity_error
    })


multidegree_anchor_results = pd.DataFrame(
    anchor_rows
)


display(
    multidegree_anchor_results
)

,component,degrees,step,target_norm,coarse_error,fine_error,richardson_error,coarse_fine_deviation,antihermiticity_error
0,L_110,"(1, 1, 0)",0.004,8.060042,1.513368e-10,1.712152e-11,4.067190e-11,1.414908e-10,0.0
1,L_210,"(2, 1, 0)",0.020,5.625253,3.120551e-04,7.800447e-05,1.840149e-08,2.340506e-04,0.0
2,L_120,"(1, 2, 0)",0.020,5.863680,3.162838e-04,7.906222e-05,1.745313e-08,2.372215e-04,0.0
3,L_111,"(1, 1, 1)",0.010,9.835250,2.878988e-05,7.197061e-06,5.339932e-09,2.159282e-05,0.0


In [8]:
# ------------------------------------------------------------------
# Acceptance tests
# ------------------------------------------------------------------

assert (
    multidegree_anchor_results[
        "coarse_error"
    ].max()
    < 2e-3
)


assert (
    multidegree_anchor_results[
        "richardson_error"
    ].max()
    < 1e-6
)


assert (
    multidegree_anchor_results[
        "antihermiticity_error"
    ].max()
    < 1e-10
)


print(
    "All analytic multidegree anchors: PASS"
)


multidegree_anchor_results.to_csv(
    result_dir
    / "multidegree_analytic_anchors.csv",
    index=False
)

All analytic multidegree anchors: PASS


双 ϵ / Richardson 收敛

In [9]:
# ------------------------------------------------------------------
# Convergence discipline and logarithm branch diagnostics
# ------------------------------------------------------------------

convergence_specs = {

    "L_210": {
        "degrees": (2, 1, 0),
        "target": multidegree_targets["L_210"]
    },

    "L_111": {
        "degrees": (1, 1, 1),
        "target": multidegree_targets["L_111"]
    }
}


step_values = np.array([
    0.08,
    0.04,
    0.02,
    0.01
])


def stencil_nodes_for_degree(degree):
    """
    Nodes used by the frozen degree-0/1/2 stencils.
    Only used here to inspect every logarithm evaluation point.
    """

    if degree == 0:
        return [0]

    if degree == 1:
        return [+1, -1]

    if degree == 2:
        return [+1, 0, -1]

    raise ValueError(
        "Only degrees 0, 1, and 2 are supported."
    )


def minimum_stencil_branch_distance(
    hamiltonians,
    degrees,
    step
):
    """
    Inspect all unitaries entering one tensor-product stencil.

    Returns:
        minimum distance of any eigenphase from +/- pi,
        maximum absolute eigenphase encountered.
    """

    branch_distances = []
    maximum_phases = []

    node_sets = [
        stencil_nodes_for_degree(degree)
        for degree in degrees
    ]


    for nodes in product(*node_sets):

        pulse_areas = [
            node * step
            for node in nodes
        ]

        U = cf.protocol_unitary(
            hamiltonians,
            pulse_areas
        )

        (
            branch_distance,
            maximum_phase
        ) = cf.unitary_branch_distance(U)

        branch_distances.append(
            branch_distance
        )

        maximum_phases.append(
            maximum_phase
        )


    return (
        min(branch_distances),
        max(maximum_phases)
    )


convergence_rows = []


for component_name, specification in convergence_specs.items():

    degrees = specification["degrees"]
    target = specification["target"]


    for step in step_values:

        (
            richardson,
            coarse,
            fine,
            coarse_fine_deviation
        ) = cf.richardson_component(
            reference_hamiltonians,
            degrees,
            step
        )


        (
            minimum_branch_distance,
            maximum_eigenphase
        ) = minimum_stencil_branch_distance(
            reference_hamiltonians,
            degrees,
            step
        )


        convergence_rows.append({

            "component": component_name,

            "degrees": str(degrees),

            "step": step,

            "coarse_error":
                relative_operator_error(
                    coarse,
                    target
                ),

            "fine_error":
                relative_operator_error(
                    fine,
                    target
                ),

            "richardson_error":
                relative_operator_error(
                    richardson,
                    target
                ),

            "coarse_fine_deviation":
                coarse_fine_deviation,

            "minimum_branch_distance":
                minimum_branch_distance,

            "maximum_abs_eigenphase":
                maximum_eigenphase
        })


convergence_results = pd.DataFrame(
    convergence_rows
)

In [10]:
convergence_results[
    "coarse_order"
] = np.nan

convergence_results[
    "richardson_order"
] = np.nan


for component_name in convergence_specs:

    indices = (
        convergence_results[
            convergence_results[
                "component"
            ] == component_name
        ]
        .sort_values(
            "step",
            ascending=False
        )
        .index
        .to_list()
    )


    for current_index, next_index in zip(
        indices[:-1],
        indices[1:]
    ):

        coarse_order = np.log2(

            convergence_results.loc[
                current_index,
                "coarse_error"
            ]

            /

            convergence_results.loc[
                next_index,
                "coarse_error"
            ]
        )


        richardson_order = np.log2(

            convergence_results.loc[
                current_index,
                "richardson_error"
            ]

            /

            convergence_results.loc[
                next_index,
                "richardson_error"
            ]
        )


        convergence_results.loc[
            current_index,
            "coarse_order"
        ] = coarse_order


        convergence_results.loc[
            current_index,
            "richardson_order"
        ] = richardson_order

In [11]:
display(
    convergence_results[
        [
            "component",
            "step",
            "coarse_error",
            "richardson_error",
            "coarse_order",
            "richardson_order",
            "minimum_branch_distance",
            "maximum_abs_eigenphase"
        ]
    ]
)

,component,step,coarse_error,richardson_error,coarse_order,richardson_order,minimum_branch_distance,maximum_abs_eigenphase
0,L_210,0.08,0.005005,4.708051e-06,2.002721,4.000939,2.856811,0.284782
1,L_210,0.04,0.001249,2.940618e-07,2.000686,3.998225,2.999015,0.142577
2,L_210,0.02,0.000312,1.840149e-08,2.000172,2.849065,3.070281,0.071312
3,L_210,0.01,0.000078,2.553871e-09,NaN,NaN,3.105934,0.035659
4,L_111,0.08,0.001854,2.238190e-05,2.007248,4.004993,2.764199,0.377394
5,L_111,0.04,0.000461,1.394036e-06,2.001589,4.001260,2.953265,0.188328
6,L_111,0.02,0.000115,8.705118e-08,2.000383,4.026971,3.047598,0.093995
7,L_111,0.01,0.000029,5.339932e-09,NaN,NaN,3.094647,0.046946


In [14]:
# ------------------------------------------------------------------
# Acceptance tests: asymptotic convergence before round-off floor
# ------------------------------------------------------------------

measured_orders = convergence_results.dropna(
    subset=[
        "coarse_order",
        "richardson_order"
    ]
).copy()


# 1. Raw centered stencils should show O(h^2)
# throughout the tested range.

assert (
    measured_orders[
        "coarse_order"
    ]
    .between(
        1.9,
        2.1
    )
    .all()
)


# 2. Richardson should show approximately O(h^4)
# in the pre-round-off window.
#
# We use h >= 0.02 because the smallest interval may already
# be dominated by floating-point cancellation after extrapolation.

richardson_asymptotic = measured_orders[
    measured_orders["step"] >= 0.02
]


assert (
    richardson_asymptotic[
        "richardson_order"
    ].median()
    > 3.5
)


# Each component must have at least one clearly fourth-order interval.

best_richardson_orders = (
    measured_orders
    .groupby("component")[
        "richardson_order"
    ]
    .max()
)


assert (
    best_richardson_orders
    > 3.5
).all()


# 3. Richardson must materially improve on the raw fine estimate
# at a stable reference step.

reference_step = 0.02

reference_rows = convergence_results[
    np.isclose(
        convergence_results["step"],
        reference_step
    )
]


assert (
    reference_rows[
        "richardson_error"
    ]
    <
    reference_rows[
        "fine_error"
    ]
).all()


assert (
    (
        reference_rows[
            "richardson_error"
        ]
        /
        reference_rows[
            "fine_error"
        ]
    ).max()
    < 1e-2
)


# 4. Every stencil evaluation must remain far from
# the principal-log branch cut.

assert (
    convergence_results[
        "minimum_branch_distance"
    ].min()
    > 1.0
)


# 5. The best Richardson result for each component
# must be highly accurate.

best_richardson_errors = (
    convergence_results
    .groupby("component")[
        "richardson_error"
    ]
    .min()
)


assert (
    best_richardson_errors
    < 1e-8
).all()


print(
    "O(h^2) raw convergence: PASS"
)

print(
    "O(h^4) Richardson asymptotic window: PASS"
)

print(
    "Richardson improvement over fine stencil: PASS"
)

print(
    "Principal-log branch safety: PASS"
)

print()

print(
    "Best Richardson errors:"
)

display(
    best_richardson_errors
    .rename("best_richardson_error")
    .reset_index()
)


convergence_results.to_csv(
    result_dir
    / "multidegree_convergence_and_branch.csv",
    index=False
)

O(h^2) raw convergence: PASS
O(h^4) Richardson asymptotic window: PASS
Richardson improvement over fine stencil: PASS
Principal-log branch safety: PASS

Best Richardson errors:


,component,best_richardson_error
0,L_111,5.339932e-09
1,L_210,2.553871e-09


qubit weight decomposition

In [15]:
# ------------------------------------------------------------------
# Basis-free support decomposition:
# cross-check against qubit Pauli strings
# ------------------------------------------------------------------

def pauli_weight_fractions(
    operator,
    n_sites,
    decomposition_tol=1e-13
):
    """
    Weight fractions computed from Pauli coefficients.

    Since
        ||O||_HS^2 = 2^N sum_P |c_P|^2,
    the common factor 2^N cancels in normalized fractions.
    """

    decomposition = cf.pauli_decomposition(
        operator,
        n_sites,
        tol=decomposition_tol
    )

    weights = {
        weight: 0.0
        for weight in range(n_sites + 1)
    }


    for _, row in decomposition.iterrows():

        coefficient_squared = (
            row["coefficient_real"] ** 2
            +
            row["coefficient_imag"] ** 2
        )

        weights[
            int(row["support_size"])
        ] += coefficient_squared


    total = sum(weights.values())

    return {
        weight: value / max(total, 1e-300)
        for weight, value in weights.items()
    }

In [16]:
# Richardson-extracted effective components

L_210_R, _, _, _ = cf.richardson_component(
    reference_hamiltonians,
    (2, 1, 0),
    0.02
)

L_111_R, _, _, _ = cf.richardson_component(
    reference_hamiltonians,
    (1, 1, 1),
    0.01
)


# Convert anti-Hermitian L coefficients to Hermitian K coefficients

K_210_R = 1j * L_210_R
K_111_R = 1j * L_111_R


# A generic Hermitian operator containing mixed weights

validation_rng = np.random.default_rng(
    20260806
)

random_matrix = (
    validation_rng.normal(
        size=(2**N, 2**N)
    )
    +
    1j
    * validation_rng.normal(
        size=(2**N, 2**N)
    )
)

random_hermitian = (
    random_matrix
    +
    random_matrix.conj().T
) / 2


validation_operators = {

    "one-body X_A":
        cf.one_body(
            cf.X,
            A,
            N
        ),

    "two-body X_A Y_B":
        cf.two_body(
            cf.X,
            cf.Y,
            A,
            B,
            N
        ),

    "three-body X_A Y_B Z_C":
        cf.kron_all([
            cf.X,
            cf.Y,
            cf.Z
        ]),

    "random Hermitian":
        random_hermitian,

    "Richardson K_210":
        K_210_R,

    "Richardson K_111":
        K_111_R
}

In [17]:
weight_validation_rows = []


for operator_name, operator in validation_operators.items():

    projector_weights = cf.weight_decomposition(
        operator,
        local_dim=2,
        n_sites=N
    )

    pauli_weights = pauli_weight_fractions(
        operator,
        n_sites=N
    )


    for weight in range(N + 1):

        weight_validation_rows.append({

            "operator": operator_name,

            "weight": weight,

            "projector_fraction":
                projector_weights[weight],

            "pauli_fraction":
                pauli_weights[weight],

            "absolute_difference":
                abs(
                    projector_weights[weight]
                    -
                    pauli_weights[weight]
                )
        })


weight_validation_results = pd.DataFrame(
    weight_validation_rows
)


display(
    weight_validation_results
)

,operator,weight,projector_fraction,pauli_fraction,absolute_difference
0,one-body X_A,0,0.000000e+00,0.000000e+00,0.000000e+00
1,one-body X_A,1,1.000000e+00,1.000000e+00,0.000000e+00
2,one-body X_A,2,0.000000e+00,0.000000e+00,0.000000e+00
3,one-body X_A,3,0.000000e+00,0.000000e+00,0.000000e+00
4,two-body X_A Y_B,0,0.000000e+00,0.000000e+00,0.000000e+00
5,two-body X_A Y_B,1,0.000000e+00,0.000000e+00,0.000000e+00
6,two-body X_A Y_B,2,1.000000e+00,1.000000e+00,0.000000e+00
7,two-body X_A Y_B,3,0.000000e+00,0.000000e+00,0.000000e+00
8,three-body X_A Y_B Z_C,0,0.000000e+00,0.000000e+00,0.000000e+00
9,three-body X_A Y_B Z_C,1,0.000000e+00,0.000000e+00,0.000000e+00


In [18]:
weight_validation_summary = (
    weight_validation_results
    .groupby(
        "operator",
        as_index=False
    )
    .agg(

        maximum_difference=(
            "absolute_difference",
            "max"
        ),

        dominant_projector_weight=(
            "projector_fraction",
            lambda values:
                int(np.argmax(
                    np.asarray(values)
                ))
        )
    )
)


display(
    weight_validation_summary
)

,operator,maximum_difference,dominant_projector_weight
0,Richardson K_111,6.060314e-27,2
1,Richardson K_210,1.708803e-27,2
2,one-body X_A,0.000000e+00,1
3,random Hermitian,1.110223e-16,3
4,three-body X_A Y_B Z_C,0.000000e+00,3
5,two-body X_A Y_B,0.000000e+00,2


In [19]:
# ------------------------------------------------------------------
# Acceptance tests
# ------------------------------------------------------------------

assert (
    weight_validation_results[
        "absolute_difference"
    ].max()
    < 1e-12
)


# Known exact-support examples

def projector_weights_for(name):

    operator = validation_operators[name]

    return cf.weight_decomposition(
        operator,
        local_dim=2,
        n_sites=N
    )


assert (
    projector_weights_for(
        "one-body X_A"
    )[1]
    > 1 - 1e-12
)


assert (
    projector_weights_for(
        "two-body X_A Y_B"
    )[2]
    > 1 - 1e-12
)


assert (
    projector_weights_for(
        "three-body X_A Y_B Z_C"
    )[3]
    > 1 - 1e-12
)


print(
    "Basis-free vs Pauli weight decomposition: PASS"
)

print(
    "Exact one-/two-/three-body support tests: PASS"
)


weight_validation_results.to_csv(
    result_dir
    / "basis_free_weight_validation.csv",
    index=False
)

Basis-free vs Pauli weight decomposition: PASS
Exact one-/two-/three-body support tests: PASS


三阶生成元的 path/cycle 正典重构（reconstruction target）

In [20]:
# ------------------------------------------------------------------
# Canonical reconstruction of the full cubic generator
# ------------------------------------------------------------------

reconstruction_step = 0.02


def equal_step_protocol_log(
    hamiltonians,
    epsilon
):
    """
    Chronological equal-step protocol:

        H1 -> H2 -> ... -> Hn

    with every pulse area equal to epsilon.
    """

    return cf.protocol_log_eig(
        hamiltonians,
        [epsilon] * len(hamiltonians)
    )


def direct_cubic_coefficient(
    hamiltonians,
    step
):
    """
    Coefficient of epsilon^3 in

        L(epsilon)
        = log U(epsilon, ..., epsilon).

    The five-point odd stencil first approximates L'''(0);
    division by 3! returns the Taylor coefficient.
    """

    L_plus_2 = equal_step_protocol_log(
        hamiltonians,
        +2 * step
    )

    L_plus_1 = equal_step_protocol_log(
        hamiltonians,
        +step
    )

    L_minus_1 = equal_step_protocol_log(
        hamiltonians,
        -step
    )

    L_minus_2 = equal_step_protocol_log(
        hamiltonians,
        -2 * step
    )


    return (
        L_plus_2
        - 2 * L_plus_1
        + 2 * L_minus_1
        - L_minus_2
    ) / (12 * step**3)


def direct_cubic_richardson(
    hamiltonians,
    step
):
    """
    Richardson extrapolation of the direct cubic coefficient.

    The centered cubic stencil has O(step^2) contamination.
    """

    coarse = direct_cubic_coefficient(
        hamiltonians,
        step
    )

    fine = direct_cubic_coefficient(
        hamiltonians,
        step / 2
    )

    richardson = (
        4 * fine - coarse
    ) / 3

    return richardson, coarse, fine

In [21]:
# Six directed wedge sectors of total degree three

wedge_degrees = [
    (2, 1, 0),
    (1, 2, 0),

    (2, 0, 1),
    (1, 0, 2),

    (0, 2, 1),
    (0, 1, 2)
]


wedge_components = {}


for degrees in wedge_degrees:

    component, _, _, _ = cf.richardson_component(
        reference_hamiltonians,
        degrees,
        reconstruction_step
    )

    wedge_components[
        degrees
    ] = component


L_path = sum(
    wedge_components.values(),
    np.zeros_like(H1)
)

In [22]:
L_cycle, _, _, _ = cf.richardson_component(
    reference_hamiltonians,
    (1, 1, 1),
    reconstruction_step
)

In [23]:
(
    L_cubic_direct,
    L_cubic_coarse,
    L_cubic_fine
) = direct_cubic_richardson(
    reference_hamiltonians,
    reconstruction_step
)


L_cubic_reconstructed = (
    L_path
    +
    L_cycle
)

In [24]:
reconstruction_absolute_error = np.linalg.norm(
    L_cubic_direct
    -
    L_cubic_reconstructed
)


reconstruction_relative_error = (
    reconstruction_absolute_error
    /
    max(
        np.linalg.norm(L_cubic_direct),
        1e-300
    )
)


direct_coarse_fine_deviation = (
    np.linalg.norm(
        L_cubic_coarse
        -
        L_cubic_fine
    )
    /
    max(
        np.linalg.norm(L_cubic_direct),
        1e-300
    )
)

In [25]:
K_path = 1j * L_path
K_cycle = 1j * L_cycle

K_cubic_direct = (
    1j * L_cubic_direct
)

K_cubic_reconstructed = (
    1j * L_cubic_reconstructed
)


path_weights = cf.weight_decomposition(
    K_path,
    local_dim=2,
    n_sites=N
)

cycle_weights = cf.weight_decomposition(
    K_cycle,
    local_dim=2,
    n_sites=N
)

total_weights = cf.weight_decomposition(
    K_cubic_direct,
    local_dim=2,
    n_sites=N
)

In [26]:
cubic_reconstruction_summary = pd.DataFrame({

    "quantity": [
        "||L_path||",
        "||L_cycle||",
        "||L_cubic_direct||",
        "||L_path + L_cycle||",
        "relative reconstruction error",
        "direct coarse/fine deviation",
        "path weight-2 fraction",
        "cycle weight-2 fraction",
        "total weight-2 fraction"
    ],

    "value": [
        np.linalg.norm(L_path),
        np.linalg.norm(L_cycle),
        np.linalg.norm(L_cubic_direct),
        np.linalg.norm(L_cubic_reconstructed),
        reconstruction_relative_error,
        direct_coarse_fine_deviation,
        path_weights[2],
        cycle_weights[2],
        total_weights[2]
    ]
})


display(
    cubic_reconstruction_summary
)

,quantity,value
0,||L_path||,1.525928e+01
1,||L_cycle||,9.835251e+00
2,||L_cubic_direct||,2.066999e+01
3,||L_path + L_cycle||,2.066997e+01
4,relative reconstruction error,9.149670e-07
5,direct coarse/fine deviation,1.243374e-03
6,path weight-2 fraction,1.000000e+00
7,cycle weight-2 fraction,1.000000e+00
8,total weight-2 fraction,1.000000e+00


In [27]:
wedge_component_summary = pd.DataFrame([

    {
        "degrees": str(degrees),
        "norm": np.linalg.norm(component),
        "weight_2_fraction":
            cf.weight_decomposition(
                1j * component,
                local_dim=2,
                n_sites=N
            )[2]
    }

    for degrees, component
    in wedge_components.items()
])


display(
    wedge_component_summary
)

,degrees,norm,weight_2_fraction
0,"(2, 1, 0)",5.625253,1.0
1,"(1, 2, 0)",5.863680,1.0
2,"(2, 0, 1)",5.965228,1.0
3,"(1, 0, 2)",7.093524,1.0
4,"(0, 2, 1)",4.724353,1.0
5,"(0, 1, 2)",5.944610,1.0


In [29]:
# ------------------------------------------------------------------
# Acceptance tests
# ------------------------------------------------------------------

# Direct cubic extraction is numerically more cancellation-sensitive
# than the tensor-product multidegree stencils.
#
# Therefore require:
#   1. a small absolute relative residual;
#   2. a residual far below the direct stencil's own coarse/fine drift.

reconstruction_to_drift_ratio = (
    reconstruction_relative_error
    /
    max(
        direct_coarse_fine_deviation,
        1e-300
    )
)


assert (
    reconstruction_relative_error
    < 2e-6
)


assert (
    reconstruction_to_drift_ratio
    < 1e-2
)


assert (
    path_weights[2]
    > 1 - 1e-10
)


assert (
    cycle_weights[2]
    > 1 - 1e-10
)


assert (
    total_weights[2]
    > 1 - 1e-10
)


assert (
    wedge_component_summary[
        "weight_2_fraction"
    ].min()
    > 1 - 1e-10
)


print(
    "Direct cubic extraction: PASS"
)

print(
    "Path + cycle reconstruction: PASS"
)

print(
    "All cubic sectors are weight-2: PASS"
)

print()

print(
    "Relative reconstruction error:",
    f"{reconstruction_relative_error:.3e}"
)

print(
    "Reconstruction / direct-drift ratio:",
    f"{reconstruction_to_drift_ratio:.3e}"
)


cubic_reconstruction_summary.to_csv(
    result_dir
    / "cubic_path_cycle_reconstruction.csv",
    index=False
)


wedge_component_summary.to_csv(
    result_dir
    / "directed_wedge_components.csv",
    index=False
)

Direct cubic extraction: PASS
Path + cycle reconstruction: PASS
All cubic sectors are weight-2: PASS

Relative reconstruction error: 9.150e-07
Reconstruction / direct-drift ratio: 7.359e-04


pair-resolved phantom support law

In [30]:
# ------------------------------------------------------------------
# Pair-resolved support law for the six directed wedge sectors
# ------------------------------------------------------------------

edge_order = [
    "AB",
    "BC",
    "AC"
]


edge_sites = {
    "AB": {A, B},
    "BC": {B, C},
    "AC": {A, C}
}


def exact_pair_fraction(
    operator,
    pair_name
):
    """
    Squared Hilbert-Schmidt fraction carried by one exact pair sector.
    """

    component = cf.exact_support_component(
        operator,
        edge_sites[pair_name],
        local_dim=2,
        n_sites=N
    )

    total_norm_squared = (
        np.linalg.norm(operator) ** 2
    )

    return (
        np.linalg.norm(component) ** 2
        /
        max(
            total_norm_squared,
            1e-300
        )
    )


wedge_pair_rows = []


for degrees, L_component in wedge_components.items():

    K_component = 1j * L_component


    repeated_index = degrees.index(2)
    single_index = degrees.index(1)
    gap_index = degrees.index(0)


    repeated_edge = edge_order[
        repeated_index
    ]

    single_edge = edge_order[
        single_index
    ]

    gap_edge = edge_order[
        gap_index
    ]


    repeated_fraction = exact_pair_fraction(
        K_component,
        repeated_edge
    )

    single_fraction = exact_pair_fraction(
        K_component,
        single_edge
    )

    gap_fraction = exact_pair_fraction(
        K_component,
        gap_edge
    )


    wedge_pair_rows.append({

        "degrees":
            str(degrees),

        "repeated_edge":
            repeated_edge,

        "single_edge":
            single_edge,

        "gap_edge":
            gap_edge,

        "repeated_edge_fraction":
            repeated_fraction,

        "single_edge_fraction":
            single_fraction,

        "phantom_gap_fraction":
            gap_fraction,

        "single_plus_gap":
            single_fraction
            +
            gap_fraction,

        "component_norm":
            np.linalg.norm(
                K_component
            )
    })


wedge_pair_support = pd.DataFrame(
    wedge_pair_rows
)


display(
    wedge_pair_support
)

,degrees,repeated_edge,single_edge,gap_edge,repeated_edge_fraction,single_edge_fraction,phantom_gap_fraction,single_plus_gap,component_norm
0,"(2, 1, 0)",AB,BC,AC,1.001840e-17,0.622650,0.377350,1.0,5.625253
1,"(1, 2, 0)",BC,AB,AC,1.067635e-16,0.584664,0.415336,1.0,5.863680
2,"(2, 0, 1)",AB,AC,BC,2.941421e-16,0.622868,0.377132,1.0,5.965228
3,"(1, 0, 2)",AC,AB,BC,2.704393e-16,0.557844,0.442156,1.0,7.093524
4,"(0, 2, 1)",BC,AC,AB,3.096643e-16,0.724474,0.275526,1.0,4.724353
5,"(0, 1, 2)",AC,BC,AB,2.089484e-17,0.589392,0.410608,1.0,5.944610


In [31]:
# ------------------------------------------------------------------
# Aggregate pair-resolved ledger
# ------------------------------------------------------------------

aggregate_operators = {

    "path sector":
        K_path,

    "cycle sector":
        K_cycle,

    "full cubic sector":
        K_cubic_direct
}


aggregate_pair_rows = []


for source_name, operator in aggregate_operators.items():

    fractions = {
        pair_name:
            exact_pair_fraction(
                operator,
                pair_name
            )

        for pair_name in edge_order
    }


    aggregate_pair_rows.append({

        "source":
            source_name,

        "AB_fraction":
            fractions["AB"],

        "BC_fraction":
            fractions["BC"],

        "AC_fraction":
            fractions["AC"],

        "pair_fraction_sum":
            sum(
                fractions.values()
            ),

        "operator_norm":
            np.linalg.norm(
                operator
            )
    })


aggregate_pair_support = pd.DataFrame(
    aggregate_pair_rows
)


display(
    aggregate_pair_support
)

,source,AB_fraction,BC_fraction,AC_fraction,pair_fraction_sum,operator_norm
0,path sector,0.343666,0.375333,0.281001,1.0,15.259279
1,cycle sector,0.322614,0.638516,0.038870,1.0,9.835251
2,full cubic sector,0.274157,0.553740,0.172103,1.0,20.669986


In [34]:
# ------------------------------------------------------------------
# Acceptance tests
# ------------------------------------------------------------------

maximum_repeated_fraction = (
    wedge_pair_support[
        "repeated_edge_fraction"
    ].max()
)


maximum_pair_closure_error = (

    wedge_pair_support[
        "single_plus_gap"
    ]

    - 1.0

).abs().max()


minimum_surviving_fraction = (
    wedge_pair_support[
        [
            "single_edge_fraction",
            "phantom_gap_fraction"
        ]
    ]
    .to_numpy()
    .min()
)


aggregate_closure_error = (

    aggregate_pair_support[
        "pair_fraction_sum"
    ]

    - 1.0

).abs().max()


assert (
    maximum_repeated_fraction
    < 1e-12
)


assert (
    maximum_pair_closure_error
    < 1e-10
)


# For this fixed generic reference sample,
# both allowed output channels should be active.

assert (
    minimum_surviving_fraction
    > 1e-6
)


assert (
    aggregate_closure_error
    < 1e-10
)


print(
    "Repeated-edge extinction: PASS"
)

print(
    "Single-edge + phantom-gap closure: PASS"
)

print(
    "Both allowed channels active for the generic sample: PASS"
)

print(
    "Aggregate pair-sector closure: PASS"
)

print()

print(
    "Maximum repeated-edge fraction:",
    f"{maximum_repeated_fraction:.3e}"
)

print(
    "Maximum wedge closure error:",
    f"{maximum_pair_closure_error:.3e}"
)


wedge_pair_support.to_csv(
    result_dir
    / "directed_wedge_pair_support.csv",
    index=False
)


aggregate_pair_support.to_csv(
    result_dir
    / "aggregate_cubic_pair_support.csv",
    index=False
)

Repeated-edge extinction: PASS
Single-edge + phantom-gap closure: PASS
Both allowed channels active for the generic sample: PASS
Aggregate pair-sector closure: PASS

Maximum repeated-edge fraction: 3.097e-16
Maximum wedge closure error: 4.441e-16


Open wedge generates phantom edge

In [35]:
# ------------------------------------------------------------------
# Open microscopic wedge: AB -- BC, with no microscopic AC edge
# ------------------------------------------------------------------

H_open_micro = (
    H_AB
    +
    H_BC
)

open_hamiltonians = [
    H_AB,
    H_BC
]


# Confirm the microscopic pair structure

microscopic_pair_rows = []


for pair_name in edge_order:

    pair_component = cf.exact_support_component(
        H_open_micro,
        edge_sites[pair_name],
        local_dim=2,
        n_sites=N
    )

    microscopic_pair_rows.append({

        "pair":
            pair_name,

        "component_norm":
            np.linalg.norm(
                pair_component
            ),

        "relative_norm":
            np.linalg.norm(
                pair_component
            )
            /
            np.linalg.norm(
                H_open_micro
            ),

        "microscopic_edge":
            pair_name in {
                "AB",
                "BC"
            }
    })


microscopic_pair_structure = pd.DataFrame(
    microscopic_pair_rows
)


display(
    microscopic_pair_structure
)

,pair,component_norm,relative_norm,microscopic_edge
0,AB,4.994867,0.712416,True
1,BC,4.920132,0.701757,True
2,AC,0.000000,0.000000,False


In [36]:
# ------------------------------------------------------------------
# Directed cubic sectors
# ------------------------------------------------------------------

open_wedge_specs = {

    "K_210: AB repeated, BC single": {

        "degrees":
            (2, 1),

        "repeated_edge":
            "AB",

        "single_edge":
            "BC",

        "gap_edge":
            "AC",

        "analytic_target":
            -1 / 12
            * cf.commutator(
                H_AB,
                cf.commutator(
                    H_AB,
                    H_BC
                )
            )
    },

    "K_120: BC repeated, AB single": {

        "degrees":
            (1, 2),

        "repeated_edge":
            "BC",

        "single_edge":
            "AB",

        "gap_edge":
            "AC",

        "analytic_target":
            +1 / 12
            * cf.commutator(
                H_BC,
                cf.commutator(
                    H_AB,
                    H_BC
                )
            )
    }
}


open_wedge_components = {}
open_wedge_rows = []


for sector_name, specification in open_wedge_specs.items():

    (
        L_component,
        _,
        _,
        extraction_deviation
    ) = cf.richardson_component(
        open_hamiltonians,
        specification["degrees"],
        step=0.02
    )


    K_component = (
        1j * L_component
    )


    repeated_edge = specification[
        "repeated_edge"
    ]

    single_edge = specification[
        "single_edge"
    ]

    gap_edge = specification[
        "gap_edge"
    ]


    pair_fractions = {

        pair_name:
            exact_pair_fraction(
                K_component,
                pair_name
            )

        for pair_name in edge_order
    }


    analytic_error = relative_operator_error(
        K_component,
        specification[
            "analytic_target"
        ]
    )


    open_wedge_components[
        sector_name
    ] = K_component


    open_wedge_rows.append({

        "sector":
            sector_name,

        "degrees":
            str(
                specification[
                    "degrees"
                ]
            ),

        "repeated_edge":
            repeated_edge,

        "single_edge":
            single_edge,

        "phantom_gap_edge":
            gap_edge,

        "AB_fraction":
            pair_fractions["AB"],

        "BC_fraction":
            pair_fractions["BC"],

        "AC_phantom_fraction":
            pair_fractions["AC"],

        "repeated_edge_fraction":
            pair_fractions[
                repeated_edge
            ],

        "single_edge_fraction":
            pair_fractions[
                single_edge
            ],

        "pair_fraction_sum":
            sum(
                pair_fractions.values()
            ),

        "component_norm":
            np.linalg.norm(
                K_component
            ),

        "analytic_error":
            analytic_error,

        "coarse_fine_deviation":
            extraction_deviation
    })


open_wedge_phantom_results = pd.DataFrame(
    open_wedge_rows
)


display(
    open_wedge_phantom_results
)

,sector,degrees,repeated_edge,single_edge,phantom_gap_edge,AB_fraction,BC_fraction,AC_phantom_fraction,repeated_edge_fraction,single_edge_fraction,pair_fraction_sum,component_norm,analytic_error,coarse_fine_deviation
0,"K_210: AB repeated, BC single","(2, 1)",AB,BC,AC,1.001840e-17,6.226496e-01,0.377350,1.001840e-17,0.622650,1.0,5.625253,1.840149e-08,0.000234
1,"K_120: BC repeated, AB single","(1, 2)",BC,AB,AC,5.846642e-01,1.067635e-16,0.415336,1.067635e-16,0.584664,1.0,5.863680,1.745313e-08,0.000237


In [37]:
# ------------------------------------------------------------------
# Acceptance tests
# ------------------------------------------------------------------

microscopic_AC_relative_norm = (

    microscopic_pair_structure
    .loc[
        microscopic_pair_structure[
            "pair"
        ] == "AC",
        "relative_norm"
    ]
    .iloc[0]
)


minimum_phantom_fraction = (
    open_wedge_phantom_results[
        "AC_phantom_fraction"
    ].min()
)


maximum_repeated_fraction = (
    open_wedge_phantom_results[
        "repeated_edge_fraction"
    ].max()
)


maximum_pair_closure_error = (

    open_wedge_phantom_results[
        "pair_fraction_sum"
    ]

    - 1.0

).abs().max()


maximum_analytic_error = (
    open_wedge_phantom_results[
        "analytic_error"
    ].max()
)


# AC is absent from the microscopic Hamiltonian.

assert (
    microscopic_AC_relative_norm
    < 1e-12
)


# Both directed wedge sectors generate a nonzero AC interaction.

assert (
    minimum_phantom_fraction
    > 1e-6
)


# The repeated microscopic edge is absent from its own directed sector.

assert (
    maximum_repeated_fraction
    < 1e-12
)


# The cubic operators close entirely inside pairwise support.

assert (
    maximum_pair_closure_error
    < 1e-10
)


# Numerical extraction agrees with the BCH nested-commutator formula.

assert (
    maximum_analytic_error
    < 1e-6
)


print(
    "Microscopic AC edge absent: PASS"
)

print(
    "Effective AC phantom edge generated: PASS"
)

print(
    "Directed repeated-edge extinction: PASS"
)

print(
    "Pair-sector closure: PASS"
)

print(
    "Analytic BCH agreement: PASS"
)

print()

print(
    "Microscopic AC relative norm:",
    f"{microscopic_AC_relative_norm:.3e}"
)

print(
    "Minimum generated AC fraction:",
    f"{minimum_phantom_fraction:.3e}"
)


microscopic_pair_structure.to_csv(
    result_dir
    / "open_wedge_microscopic_structure.csv",
    index=False
)


open_wedge_phantom_results.to_csv(
    result_dir
    / "open_wedge_phantom_edges.csv",
    index=False
)

Microscopic AC edge absent: PASS
Effective AC phantom edge generated: PASS
Directed repeated-edge extinction: PASS
Pair-sector closure: PASS
Analytic BCH agreement: PASS

Microscopic AC relative norm: 0.000e+00
Minimum generated AC fraction: 3.774e-01


In [38]:
# ------------------------------------------------------------------
# Finite-step phantom edge:
# G_eff = i log(U) / epsilon
# ------------------------------------------------------------------

# Analytic cubic coefficient in K = iL

K_open_cubic = sum(
    open_wedge_components.values(),
    np.zeros_like(H_AB)
)


K_AC_analytic = cf.exact_support_component(
    K_open_cubic,
    {A, C},
    local_dim=2,
    n_sites=N
)


assert (
    np.linalg.norm(K_AC_analytic)
    > tol
)


epsilon_values = np.array([
    0.16,
    0.12,
    0.08,
    0.06,
    0.04,
    0.03,
    0.02
])


phantom_scaling_rows = []


for epsilon in epsilon_values:

    U_step = cf.protocol_unitary(
        [H_AB, H_BC],
        [epsilon, epsilon]
    )


    (
        branch_distance,
        maximum_abs_eigenphase
    ) = cf.unitary_branch_distance(
        U_step
    )


    L_step = cf.log_unitary(
        U_step
    )


    # Digital-step normalization:
    # one protocol cycle represents simulated time epsilon.

    G_eff = (
        1j * L_step
        / epsilon
    )


    delta_G = (
        G_eff
        -
        H_open_micro
    )


    G_AC_phantom = cf.exact_support_component(
        delta_G,
        {A, C},
        local_dim=2,
        n_sites=N
    )


    phantom_norm = np.linalg.norm(
        G_AC_phantom
    )


    analytic_norm_squared = np.vdot(
        K_AC_analytic,
        K_AC_analytic
    ).real


    # Best scalar projection onto the analytic phantom direction

    fitted_amplitude = (
        np.vdot(
            K_AC_analytic,
            G_AC_phantom
        ).real
        /
        analytic_norm_squared
    )


    fitted_component = (
        fitted_amplitude
        * K_AC_analytic
    )


    direction_residual = (
        np.linalg.norm(
            G_AC_phantom
            -
            fitted_component
        )
        /
        max(
            phantom_norm,
            1e-300
        )
    )


    scaled_analytic_error = (
        np.linalg.norm(
            G_AC_phantom / epsilon**2
            -
            K_AC_analytic
        )
        /
        np.linalg.norm(
            K_AC_analytic
        )
    )


    phantom_scaling_rows.append({

        "epsilon":
            epsilon,

        "phantom_AC_norm":
            phantom_norm,

        "fitted_amplitude":
            fitted_amplitude,

        "amplitude_over_epsilon2":
            fitted_amplitude
            / epsilon**2,

        "direction_residual":
            direction_residual,

        "scaled_analytic_error":
            scaled_analytic_error,

        "branch_distance":
            branch_distance,

        "maximum_abs_eigenphase":
            maximum_abs_eigenphase
    })


phantom_scaling_results = pd.DataFrame(
    phantom_scaling_rows
)


# Fit only the small-epsilon asymptotic window

fit_mask = (
    phantom_scaling_results[
        "epsilon"
    ]
    <= 0.08
)


phantom_scaling_slope, phantom_scaling_intercept = np.polyfit(

    np.log(
        phantom_scaling_results.loc[
            fit_mask,
            "epsilon"
        ]
    ),

    np.log(
        phantom_scaling_results.loc[
            fit_mask,
            "phantom_AC_norm"
        ]
    ),

    deg=1
)


display(
    phantom_scaling_results
)


phantom_scaling_summary = pd.DataFrame({

    "quantity": [
        "analytic phantom coefficient norm",
        "fitted epsilon exponent",
        "minimum branch distance",
        "maximum direction residual",
        "smallest-epsilon scaled analytic error",
        "smallest-epsilon amplitude / epsilon^2"
    ],

    "value": [
        np.linalg.norm(
            K_AC_analytic
        ),

        phantom_scaling_slope,

        phantom_scaling_results[
            "branch_distance"
        ].min(),

        phantom_scaling_results[
            "direction_residual"
        ].max(),

        phantom_scaling_results
        .sort_values("epsilon")
        .iloc[0][
            "scaled_analytic_error"
        ],

        phantom_scaling_results
        .sort_values("epsilon")
        .iloc[0][
            "amplitude_over_epsilon2"
        ]
    ]
})


display(
    phantom_scaling_summary
)


# ------------------------------------------------------------------
# Acceptance tests
# ------------------------------------------------------------------

assert (
    phantom_scaling_results[
        "phantom_AC_norm"
    ].min()
    > 0
)


assert (
    1.95
    <
    phantom_scaling_slope
    <
    2.05
)


assert (
    phantom_scaling_results[
        "branch_distance"
    ].min()
    > 1.0
)


assert (
    phantom_scaling_results[
        "direction_residual"
    ].max()
    < 1e-2
)


smallest_epsilon_row = (
    phantom_scaling_results
    .sort_values("epsilon")
    .iloc[0]
)


assert (
    smallest_epsilon_row[
        "scaled_analytic_error"
    ]
    < 5e-4
)


assert (
    abs(
        smallest_epsilon_row[
            "amplitude_over_epsilon2"
        ]
        - 1.0
    )
    < 5e-4
)


print(
    "Finite-step AC phantom edge: PASS"
)

print(
    "Phantom amplitude proportional to epsilon^2: PASS"
)

print(
    "Analytic phantom direction recovered: PASS"
)

print(
    "Principal-log branch safety: PASS"
)

print()

print(
    "Fitted exponent:",
    f"{phantom_scaling_slope:.6f}"
)


phantom_scaling_results.to_csv(
    result_dir
    / "finite_step_phantom_scaling.csv",
    index=False
)

,epsilon,phantom_AC_norm,fitted_amplitude,amplitude_over_epsilon2,direction_residual,scaled_analytic_error,branch_distance,maximum_abs_eigenphase
0,0.16,0.127849,0.025923,1.012602,0.003193,0.013011,2.590604,0.550988
1,0.12,0.071526,0.014503,1.007124,0.001784,0.007347,2.727095,0.414497
2,0.08,0.031664,0.006420,1.003176,0.000789,0.003273,2.864671,0.276921
3,0.06,0.017787,0.003606,1.001789,0.000443,0.001843,2.933748,0.207845
4,0.04,0.007897,0.001601,1.000795,0.000197,0.000819,3.002957,0.138636
5,0.03,0.004441,0.000900,1.000448,0.000111,0.000461,3.037597,0.103996
6,0.02,0.001973,0.000400,1.000199,0.000049,0.000205,3.072253,0.069340


,quantity,value
0,analytic phantom coefficient norm,4.931911
1,fitted epsilon exponent,2.002079
2,minimum branch distance,2.590604
3,maximum direction residual,0.003193
4,smallest-epsilon scaled analytic error,0.000205
5,smallest-epsilon amplitude / epsilon^2,1.000199


Finite-step AC phantom edge: PASS
Phantom amplitude proportional to epsilon^2: PASS
Analytic phantom direction recovered: PASS
Principal-log branch safety: PASS

Fitted exponent: 2.002079


Nested group-commutator witness

In [39]:
# ------------------------------------------------------------------
# Nested group-commutator witness
# ------------------------------------------------------------------

from scipy.linalg import expm


identity_3q = np.eye(
    2**N,
    dtype=complex
)


# Generic target channel

generic_nested_target = cf.commutator(
    H1,
    cf.commutator(
        H2,
        H3
    )
)


# Kitaev-type null configuration

H_K1 = cf.two_body(
    cf.X,
    cf.X,
    A,
    B,
    N
)

H_K2 = cf.two_body(
    cf.Y,
    cf.Y,
    B,
    C,
    N
)

H_K3 = cf.two_body(
    cf.Z,
    cf.Z,
    A,
    C,
    N
)


kitaev_pairwise_norms = np.array([
    np.linalg.norm(
        cf.commutator(
            H_K1,
            H_K2
        )
    ),

    np.linalg.norm(
        cf.commutator(
            H_K1,
            H_K3
        )
    ),

    np.linalg.norm(
        cf.commutator(
            H_K2,
            H_K3
        )
    )
])


kitaev_nested_target = cf.commutator(
    H_K1,
    cf.commutator(
        H_K2,
        H_K3
    )
)


print(
    "Generic nested-target norm:",
    np.linalg.norm(
        generic_nested_target
    )
)

print(
    "Kitaev pairwise commutator norms:",
    kitaev_pairwise_norms
)

print(
    "Kitaev nested-target norm:",
    np.linalg.norm(
        kitaev_nested_target
    )
)

Generic nested-target norm: 26.29150548653968
Kitaev pairwise commutator norms: [5.656854 5.656854 5.656854]
Kitaev nested-target norm: 0.0


In [41]:
witness_epsilon_values = np.array([
    0.20,
    0.16,
    0.12,
    0.10,
    0.08,
    0.06,
    0.05,
    0.04,
    0.03
])


witness_rows = []


for epsilon in witness_epsilon_values:

    # Generic triangle

    W_generic = cf.group_commutator_unitary(
        H1,
        H2,
        H3,
        epsilon
    )


    W_generic_target = expm(
        1j
        * epsilon**3
        * generic_nested_target
    )


    generic_signal = np.linalg.norm(
        W_generic
        -
        identity_3q
    )


    generic_target_residual = np.linalg.norm(
        W_generic
        -
        W_generic_target
    )


    generic_target_ratio = (
        generic_target_residual
        /
        max(
            generic_signal,
            1e-300
        )
    )


    # Kitaev null configuration

    W_kitaev = cf.group_commutator_unitary(
        H_K1,
        H_K2,
        H_K3,
        epsilon
    )


    kitaev_signal = np.linalg.norm(
        W_kitaev
        -
        identity_3q
    )


    witness_rows.append({

        "epsilon":
            epsilon,

        "generic_signal":
            generic_signal,

        "generic_target_residual":
            generic_target_residual,

        "generic_target_ratio":
            generic_target_ratio,

        "kitaev_null_signal":
            kitaev_signal
    })


witness_scaling_results = pd.DataFrame(
    witness_rows
)

In [42]:
witness_fit_mask = (
    witness_scaling_results[
        "epsilon"
    ]
    <= 0.10
)


def loglog_slope(
    x,
    y
):

    return np.polyfit(
        np.log(x),
        np.log(y),
        deg=1
    )[0]


generic_signal_slope = loglog_slope(

    witness_scaling_results.loc[
        witness_fit_mask,
        "epsilon"
    ],

    witness_scaling_results.loc[
        witness_fit_mask,
        "generic_signal"
    ]
)


generic_residual_slope = loglog_slope(

    witness_scaling_results.loc[
        witness_fit_mask,
        "epsilon"
    ],

    witness_scaling_results.loc[
        witness_fit_mask,
        "generic_target_residual"
    ]
)


kitaev_null_slope = loglog_slope(

    witness_scaling_results.loc[
        witness_fit_mask,
        "epsilon"
    ],

    witness_scaling_results.loc[
        witness_fit_mask,
        "kitaev_null_signal"
    ]
)


display(
    witness_scaling_results
)


witness_scaling_summary = pd.DataFrame({

    "quantity": [
        "generic nested-target norm",
        "generic signal exponent",
        "generic target-residual exponent",
        "Kitaev nested-target norm",
        "Kitaev null-signal exponent",
        "smallest-epsilon target ratio"
    ],

    "value": [
        np.linalg.norm(
            generic_nested_target
        ),

        generic_signal_slope,

        generic_residual_slope,

        np.linalg.norm(
            kitaev_nested_target
        ),

        kitaev_null_slope,

        witness_scaling_results
        .sort_values("epsilon")
        .iloc[0][
            "generic_target_ratio"
        ]
    ]
})


display(
    witness_scaling_summary
)

,epsilon,generic_signal,generic_target_residual,generic_target_ratio,kitaev_null_signal
0,0.20,0.229226,0.138932,0.606094,0.024429
1,0.16,0.115401,0.060454,0.523859,0.010177
2,0.12,0.047595,0.020084,0.421986,0.003262
3,0.10,0.027220,0.009878,0.362884,0.001581
4,0.08,0.013782,0.004112,0.298365,0.000650
5,0.06,0.005758,0.001318,0.228850,0.000206
6,0.05,0.003319,0.000639,0.192453,0.000100
7,0.04,0.001693,0.000263,0.155134,0.000041
8,0.03,0.000712,0.000083,0.117048,0.000013


,quantity,value
0,generic nested-target norm,26.291505
1,generic signal exponent,3.025435
2,generic target-residual exponent,3.966490
3,Kitaev nested-target norm,0.000000
4,Kitaev null-signal exponent,3.991410
5,smallest-epsilon target ratio,0.117048


In [43]:
# ------------------------------------------------------------------
# Acceptance tests
# ------------------------------------------------------------------

# Kitaev configuration remains pairwise noncommuting.

assert (
    kitaev_pairwise_norms.min()
    > 1e-6
)


# But the selected nested channel is exactly null.

assert (
    np.linalg.norm(
        kitaev_nested_target
    )
    < 1e-12
)


# Generic witness signal begins at cubic order.

assert (
    2.9
    <
    generic_signal_slope
    <
    3.1
)


# Difference from the analytic leading target begins at fourth order.

assert (
    3.8
    <
    generic_residual_slope
    <
    4.2
)


# In the Kitaev null configuration, the cubic signal is absent.

assert (
    3.8
    <
    kitaev_null_slope
    <
    4.2
)


# The leading-order target becomes relatively more accurate
# as epsilon decreases.

target_ratios = (
    witness_scaling_results
    .sort_values(
        "epsilon",
        ascending=False
    )[
        "generic_target_ratio"
    ]
    .to_numpy()
)


assert (
    np.diff(
        target_ratios
    )
    < 0
).all()


print(
    "Generic witness signal is O(epsilon^3): PASS"
)

print(
    "Generic target deviation is O(epsilon^4): PASS"
)

print(
    "Kitaev pairwise commutators remain nonzero: PASS"
)

print(
    "Kitaev cubic witness channel is null: PASS"
)

print(
    "Kitaev residual signal is O(epsilon^4): PASS"
)

print()

print(
    "Generic signal exponent:",
    f"{generic_signal_slope:.6f}"
)

print(
    "Generic residual exponent:",
    f"{generic_residual_slope:.6f}"
)

print(
    "Kitaev null exponent:",
    f"{kitaev_null_slope:.6f}"
)


witness_scaling_results.to_csv(
    result_dir
    / "nested_witness_scaling.csv",
    index=False
)


witness_scaling_summary.to_csv(
    result_dir
    / "nested_witness_summary.csv",
    index=False
)

Generic witness signal is O(epsilon^3): PASS
Generic target deviation is O(epsilon^4): PASS
Kitaev pairwise commutators remain nonzero: PASS
Kitaev cubic witness channel is null: PASS
Kitaev residual signal is O(epsilon^4): PASS

Generic signal exponent: 3.025435
Generic residual exponent: 3.966490
Kitaev null exponent: 3.991410


Asymmetric Trotter 的三体误差 vs palindrome 的全阶 pairwise closure

In [44]:
# ------------------------------------------------------------------
# Asymmetric Trotter vs symmetric palindrome
# ------------------------------------------------------------------

comparison_epsilon_values = np.array([
    0.20,
    0.16,
    0.12,
    0.10,
    0.08,
    0.06,
    0.04,
    0.03,
    0.02
])


# ------------------------------------------------------------------
# Homogeneous quadratic BCH term of the asymmetric protocol
# ------------------------------------------------------------------

L_asym_quadratic = (
    0.5
    * cf.commutator(
        H_AB,
        H_BC
    )
)

K_asym_quadratic = (
    1j
    * L_asym_quadratic
)


asym_quadratic_weights = cf.weight_decomposition(
    K_asym_quadratic,
    local_dim=2,
    n_sites=N
)


quadratic_weight_summary = pd.DataFrame({

    "weight":
        list(
            asym_quadratic_weights.keys()
        ),

    "fraction":
        list(
            asym_quadratic_weights.values()
        )
})


display(
    quadratic_weight_summary
)


# ------------------------------------------------------------------
# Exact finite-step logarithms
# ------------------------------------------------------------------

protocol_comparison_rows = []


for epsilon in comparison_epsilon_values:

    # --------------------------------------------------------------
    # Asymmetric first-order Trotter:
    # AB -> BC
    # --------------------------------------------------------------

    U_asym = cf.protocol_unitary(
        [H_AB, H_BC],
        [epsilon, epsilon]
    )

    L_asym = cf.log_unitary(
        U_asym
    )

    K_asym = (
        1j * L_asym
    )

    delta_K_asym = (
        K_asym
        -
        epsilon * H_open_micro
    )

    asym_correction_weights = cf.weight_decomposition(
        delta_K_asym,
        local_dim=2,
        n_sites=N
    )


    # --------------------------------------------------------------
    # Symmetric Strang palindrome:
    # AB/2 -> BC -> AB/2
    # --------------------------------------------------------------

    palindrome_hams, palindrome_areas = cf.strang_lists(
        H_AB,
        H_BC,
        epsilon
    )

    U_palindrome = cf.protocol_unitary(
        palindrome_hams,
        palindrome_areas
    )

    L_palindrome = cf.log_unitary(
        U_palindrome
    )

    K_palindrome = (
        1j * L_palindrome
    )

    delta_K_palindrome = (
        K_palindrome
        -
        epsilon * H_open_micro
    )

    palindrome_full_weights = cf.weight_decomposition(
        K_palindrome,
        local_dim=2,
        n_sites=N
    )

    palindrome_correction_weights = cf.weight_decomposition(
        delta_K_palindrome,
        local_dim=2,
        n_sites=N
    )


    # --------------------------------------------------------------
    # Explicit time-reversal check:
    # U_pal(-eps) = U_pal(eps)^\dagger
    # --------------------------------------------------------------

    reverse_hams, reverse_areas = cf.strang_lists(
        H_AB,
        H_BC,
        -epsilon
    )

    U_palindrome_negative = cf.protocol_unitary(
        reverse_hams,
        reverse_areas
    )

    time_symmetry_error = (
        np.linalg.norm(
            U_palindrome_negative
            -
            U_palindrome.conj().T
        )
        /
        np.linalg.norm(
            U_palindrome
        )
    )


    asym_branch_distance, _ = cf.unitary_branch_distance(
        U_asym
    )

    palindrome_branch_distance, _ = cf.unitary_branch_distance(
        U_palindrome
    )


    protocol_comparison_rows.append({

        "epsilon":
            epsilon,

        "asym_correction_norm":
            np.linalg.norm(
                delta_K_asym
            ),

        "asym_weight2_fraction":
            asym_correction_weights[2],

        "asym_weight3_fraction":
            asym_correction_weights[3],

        "palindrome_correction_norm":
            np.linalg.norm(
                delta_K_palindrome
            ),

        "palindrome_correction_weight2":
            palindrome_correction_weights[2],

        "palindrome_correction_weight3":
            palindrome_correction_weights[3],

        "palindrome_full_weight2":
            palindrome_full_weights[2],

        "palindrome_full_weight3":
            palindrome_full_weights[3],

        "time_symmetry_error":
            time_symmetry_error,

        "asym_branch_distance":
            asym_branch_distance,

        "palindrome_branch_distance":
            palindrome_branch_distance
    })


protocol_comparison_results = pd.DataFrame(
    protocol_comparison_rows
)


display(
    protocol_comparison_results
)

,weight,fraction
0,0,0.000000e+00
1,1,0.000000e+00
2,2,2.846014e-34
3,3,1.000000e+00


,epsilon,asym_correction_norm,asym_weight2_fraction,asym_weight3_fraction,palindrome_correction_norm,palindrome_correction_weight2,palindrome_correction_weight3,palindrome_full_weight2,palindrome_full_weight3,time_symmetry_error,asym_branch_distance,palindrome_branch_distance
0,0.20,0.329409,0.040229,0.959762,0.054172,1.0,2.598692e-28,1.0,4.056325e-31,1.512635e-16,2.455600,2.455600
1,0.16,0.209196,0.025576,0.974421,0.027446,1.0,5.958272e-28,1.0,3.573754e-31,1.502130e-16,2.590604,2.590604
2,0.12,0.116967,0.014311,0.985688,0.011484,1.0,5.272115e-27,1.0,9.845446e-31,1.636718e-16,2.727095,2.727095
3,0.10,0.081035,0.009918,0.990082,0.006624,1.0,4.304060e-27,1.0,3.973499e-31,1.294379e-16,2.795772,2.795772
4,0.08,0.051762,0.006337,0.993663,0.003383,1.0,3.374301e-26,1.0,1.238344e-30,1.075917e-16,2.864671,2.864671
5,0.06,0.029072,0.003560,0.996440,0.001424,1.0,1.543724e-25,1.0,1.784326e-30,7.583773e-17,2.933748,2.933748
6,0.04,0.012907,0.001581,0.998419,0.000421,1.0,1.219394e-24,1.0,2.762078e-30,9.326053e-17,3.002957,3.002957
7,0.03,0.007258,0.000889,0.999111,0.000178,1.0,9.986553e-24,1.0,7.097030e-30,1.260445e-16,3.037597,3.037597
8,0.02,0.003225,0.000395,0.999605,0.000053,1.0,9.045064e-23,1.0,1.274929e-29,8.875300e-17,3.072253,3.072253


In [45]:
comparison_fit_mask = (
    protocol_comparison_results[
        "epsilon"
    ]
    <= 0.08
)


asym_correction_exponent = loglog_slope(

    protocol_comparison_results.loc[
        comparison_fit_mask,
        "epsilon"
    ],

    protocol_comparison_results.loc[
        comparison_fit_mask,
        "asym_correction_norm"
    ]
)


palindrome_correction_exponent = loglog_slope(

    protocol_comparison_results.loc[
        comparison_fit_mask,
        "epsilon"
    ],

    protocol_comparison_results.loc[
        comparison_fit_mask,
        "palindrome_correction_norm"
    ]
)


protocol_comparison_summary = pd.DataFrame({

    "quantity": [
        "asymmetric quadratic weight-3 fraction",
        "asymmetric correction exponent",
        "palindrome correction exponent",
        "minimum palindrome correction weight-2",
        "maximum palindrome correction weight-3",
        "minimum palindrome full weight-2",
        "maximum time-symmetry error",
        "minimum branch distance"
    ],

    "value": [
        asym_quadratic_weights[3],

        asym_correction_exponent,

        palindrome_correction_exponent,

        protocol_comparison_results[
            "palindrome_correction_weight2"
        ].min(),

        protocol_comparison_results[
            "palindrome_correction_weight3"
        ].max(),

        protocol_comparison_results[
            "palindrome_full_weight2"
        ].min(),

        protocol_comparison_results[
            "time_symmetry_error"
        ].max(),

        min(
            protocol_comparison_results[
                "asym_branch_distance"
            ].min(),

            protocol_comparison_results[
                "palindrome_branch_distance"
            ].min()
        )
    ]
})


display(
    protocol_comparison_summary
)

,quantity,value
0,asymmetric quadratic weight-3 fraction,1.000000e+00
1,asymmetric correction exponent,2.002255e+00
2,palindrome correction exponent,3.003089e+00
3,minimum palindrome correction weight-2,1.000000e+00
4,maximum palindrome correction weight-3,9.045064e-23
5,minimum palindrome full weight-2,1.000000e+00
6,maximum time-symmetry error,1.636718e-16
7,minimum branch distance,2.455600e+00


In [46]:
# ------------------------------------------------------------------
# Acceptance tests
# ------------------------------------------------------------------

# The homogeneous second-order asymmetric error is purely three-body.

assert (
    asym_quadratic_weights[3]
    > 1 - 1e-12
)


# Exact asymmetric correction begins at epsilon^2.

assert (
    1.9
    <
    asym_correction_exponent
    <
    2.1
)


# Exact palindrome correction begins at epsilon^3.

assert (
    2.9
    <
    palindrome_correction_exponent
    <
    3.1
)


# The complete palindrome logarithm and its correction remain
# inside the pairwise sector throughout the tested branch-safe range.

assert (
    protocol_comparison_results[
        "palindrome_correction_weight2"
    ].min()
    > 1 - 1e-10
)


assert (
    protocol_comparison_results[
        "palindrome_correction_weight3"
    ].max()
    < 1e-10
)


assert (
    protocol_comparison_results[
        "palindrome_full_weight2"
    ].min()
    > 1 - 1e-10
)


# At small epsilon, the asymmetric finite-step correction
# converges to its pure weight-3 quadratic term.

smallest_epsilon_comparison = (
    protocol_comparison_results
    .sort_values("epsilon")
    .iloc[0]
)


assert (
    smallest_epsilon_comparison[
        "asym_weight3_fraction"
    ]
    > 0.995
)


# Explicit time symmetry and branch safety.

assert (
    protocol_comparison_results[
        "time_symmetry_error"
    ].max()
    < 1e-12
)


assert (
    protocol_comparison_summary
    .loc[
        protocol_comparison_summary[
            "quantity"
        ] == "minimum branch distance",
        "value"
    ]
    .iloc[0]
    > 1.0
)


print(
    "Asymmetric quadratic correction is pure weight-3: PASS"
)

print(
    "Asymmetric finite-step correction is O(epsilon^2): PASS"
)

print(
    "Palindrome finite-step correction is O(epsilon^3): PASS"
)

print(
    "Palindrome full-order pairwise closure: PASS"
)

print(
    "Palindrome time-reversal symmetry: PASS"
)

print()

print(
    "Asymmetric correction exponent:",
    f"{asym_correction_exponent:.6f}"
)

print(
    "Palindrome correction exponent:",
    f"{palindrome_correction_exponent:.6f}"
)


protocol_comparison_results.to_csv(
    result_dir
    / "asymmetric_vs_palindrome.csv",
    index=False
)


protocol_comparison_summary.to_csv(
    result_dir
    / "asymmetric_vs_palindrome_summary.csv",
    index=False
)

Asymmetric quadratic correction is pure weight-3: PASS
Asymmetric finite-step correction is O(epsilon^2): PASS
Palindrome finite-step correction is O(epsilon^3): PASS
Palindrome full-order pairwise closure: PASS
Palindrome time-reversal symmetry: PASS

Asymmetric correction exponent: 2.002255
Palindrome correction exponent: 3.003089


motif-level cycle tax

In [47]:
# ------------------------------------------------------------------
# Cycle tax:
# square -> weight 3
# pentagon -> weight 4
# all-conflict square -> exact extinction
# ------------------------------------------------------------------

def ordered_cycle_word(edge_operators):
    """
    For edges [E1, E2, ..., Em], evaluate

        [Em, [..., [E3, [E1, E2]] ...]].

    The edge list follows the order around the cycle.
    """

    if len(edge_operators) < 2:
        raise ValueError(
            "At least two edge operators are required."
        )

    output = cf.commutator(
        edge_operators[0],
        edge_operators[1]
    )

    for edge_operator in edge_operators[2:]:
        output = cf.commutator(
            edge_operator,
            output
        )

    return output


# ==================================================================
# 1. Active square:
#
#   AB = X_A X_B
#   BC = Y_B X_C
#   CD = Y_C X_D
#   DA = X_D Y_A
#
# There is exactly one matching cycle vertex, at D.
# ==================================================================

N_square = 4

A4, B4, C4, D4 = 0, 1, 2, 3


square_active_edges = [

    cf.single_edge_word(
        A4, B4,
        "X", "X",
        N_square
    ),

    cf.single_edge_word(
        B4, C4,
        "Y", "X",
        N_square
    ),

    cf.single_edge_word(
        C4, D4,
        "Y", "X",
        N_square
    ),

    cf.single_edge_word(
        D4, A4,
        "X", "Y",
        N_square
    )
]


square_raw = ordered_cycle_word(
    square_active_edges
)


# Three nested commutators of Hermitian inputs give
# an anti-Hermitian operator; multiply by i for a Hermitian channel.

square_channel = (
    1j * square_raw
)


square_weights = cf.weight_decomposition(
    square_channel,
    local_dim=2,
    n_sites=N_square
)


square_pauli = cf.pauli_decomposition(
    square_channel,
    n_sites=N_square,
    tol=1e-12
)


# ==================================================================
# 2. Active pentagon:
#
#   AB = X_A X_B
#   BC = Y_B X_C
#   CD = Y_C X_D
#   DE = Y_D X_E
#   EA = X_E Y_A
#
# Again there is one matching cycle vertex, now at E.
# ==================================================================

N_pentagon = 5

A5, B5, C5, D5, E5 = 0, 1, 2, 3, 4


pentagon_active_edges = [

    cf.single_edge_word(
        A5, B5,
        "X", "X",
        N_pentagon
    ),

    cf.single_edge_word(
        B5, C5,
        "Y", "X",
        N_pentagon
    ),

    cf.single_edge_word(
        C5, D5,
        "Y", "X",
        N_pentagon
    ),

    cf.single_edge_word(
        D5, E5,
        "Y", "X",
        N_pentagon
    ),

    cf.single_edge_word(
        E5, A5,
        "X", "Y",
        N_pentagon
    )
]


pentagon_channel = ordered_cycle_word(
    pentagon_active_edges
)


pentagon_weights = cf.weight_decomposition(
    pentagon_channel,
    local_dim=2,
    n_sites=N_pentagon
)


pentagon_pauli = cf.pauli_decomposition(
    pentagon_channel,
    n_sites=N_pentagon,
    tol=1e-12
)


# ==================================================================
# 3. All-conflict square null:
#
#   AB = X_A X_B
#   BC = Y_B X_C
#   CD = Y_C X_D
#   DA = Y_D Y_A
#
# Every cycle vertex carries a Pauli mismatch.
# All adjacent pairwise commutators remain nonzero.
# ==================================================================

square_null_edges = [

    cf.single_edge_word(
        A4, B4,
        "X", "X",
        N_square
    ),

    cf.single_edge_word(
        B4, C4,
        "Y", "X",
        N_square
    ),

    cf.single_edge_word(
        C4, D4,
        "Y", "X",
        N_square
    ),

    cf.single_edge_word(
        D4, A4,
        "Y", "Y",
        N_square
    )
]


square_null_channel = ordered_cycle_word(
    square_null_edges
)


square_null_adjacent_norms = np.array([

    np.linalg.norm(
        cf.commutator(
            square_null_edges[index],
            square_null_edges[
                (index + 1) % 4
            ]
        )
    )

    for index in range(4)
])


# ==================================================================
# Summary
# ==================================================================

cycle_tax_summary = pd.DataFrame({

    "motif": [
        "active square",
        "active pentagon",
        "all-conflict square"
    ],

    "cycle_length": [
        4,
        5,
        4
    ],

    "channel_norm": [
        np.linalg.norm(
            square_channel
        ),

        np.linalg.norm(
            pentagon_channel
        ),

        np.linalg.norm(
            square_null_channel
        )
    ],

    "dominant_weight": [
        max(
            square_weights,
            key=square_weights.get
        ),

        max(
            pentagon_weights,
            key=pentagon_weights.get
        ),

        np.nan
    ],

    "dominant_fraction": [
        max(
            square_weights.values()
        ),

        max(
            pentagon_weights.values()
        ),

        0.0
    ]
})


display(
    cycle_tax_summary
)


print(
    "Active square Pauli output:"
)

display(
    square_pauli
)


print(
    "Active pentagon Pauli output:"
)

display(
    pentagon_pauli
)


print(
    "All-conflict square adjacent commutator norms:",
    square_null_adjacent_norms
)


# ==================================================================
# Acceptance tests
# ==================================================================

assert (
    np.linalg.norm(
        square_channel
    )
    > tol
)


assert (
    square_weights[3]
    > 1 - 1e-12
)


assert (
    len(square_pauli)
    == 1
)


assert (
    square_pauli.iloc[0][
        "pauli_string"
    ]
    == "ZZZI"
)


assert (
    np.linalg.norm(
        pentagon_channel
    )
    > tol
)


assert (
    pentagon_weights[4]
    > 1 - 1e-12
)


assert (
    len(pentagon_pauli)
    == 1
)


assert (
    pentagon_pauli.iloc[0][
        "pauli_string"
    ]
    == "ZZZZI"
)


# The null is not caused by commuting adjacent edges.

assert (
    square_null_adjacent_norms.min()
    > tol
)


assert (
    np.linalg.norm(
        square_null_channel
    )
    < 1e-12
)


print()

print(
    "Square cycle produces pure weight-3: PASS"
)

print(
    "Pentagon cycle produces pure weight-4: PASS"
)

print(
    "All-conflict square has nonzero local commutators: PASS"
)

print(
    "All-conflict square cycle channel is null: PASS"
)


cycle_tax_summary.to_csv(
    result_dir
    / "cycle_tax_canonical_examples.csv",
    index=False
)


square_pauli.to_csv(
    result_dir
    / "square_cycle_pauli_output.csv",
    index=False
)


pentagon_pauli.to_csv(
    result_dir
    / "pentagon_cycle_pauli_output.csv",
    index=False
)

,motif,cycle_length,channel_norm,dominant_weight,dominant_fraction
0,active square,4,32.000000,3.0,1.0
1,active pentagon,5,90.509668,4.0,1.0
2,all-conflict square,4,0.000000,NaN,0.0


Active square Pauli output:


,pauli_string,coefficient_real,coefficient_imag,support_size
0,ZZZI,8.0,0.0,3


Active pentagon Pauli output:


,pauli_string,coefficient_real,coefficient_imag,support_size
0,ZZZZI,-16.0,0.0,4


All-conflict square adjacent commutator norms: [8. 8. 8. 8.]

Square cycle produces pure weight-3: PASS
Pentagon cycle produces pure weight-4: PASS
All-conflict square has nonzero local commutators: PASS
All-conflict square cycle channel is null: PASS


Spin-s extension

In [48]:
# ------------------------------------------------------------------
# Spin-s extension of the square-cycle examples
# ------------------------------------------------------------------

spin_dimensions = [
    2,  # spin 1/2
    3,  # spin 1
    4   # spin 3/2
]


spin_square_rows = []


for local_dim in spin_dimensions:

    spin = (
        local_dim - 1
    ) / 2


    Sx, Sy, Sz = cf.spin_ops(
        local_dim,
        normalized=True
    )


    identity_local = np.eye(
        local_dim,
        dtype=complex
    )


    # --------------------------------------------------------------
    # Local algebra diagnostics
    # --------------------------------------------------------------

    Sx_squared = (
        Sx @ Sx
    )


    Sx_squared_trivial = (
        np.trace(Sx_squared)
        / local_dim
        * identity_local
    )


    Sx_squared_traceless = (
        Sx_squared
        -
        Sx_squared_trivial
    )


    Sxy_anticommutator = (
        Sx @ Sy
        +
        Sy @ Sx
    )


    # --------------------------------------------------------------
    # Active square:
    #
    # AB = Sx_A Sx_B
    # BC = Sy_B Sx_C
    # CD = Sy_C Sx_D
    # DA = Sx_D Sy_A
    # --------------------------------------------------------------

    active_edges = [

        cf.embed_two_body(
            Sx, Sx,
            A4, B4,
            N_square,
            local_dim
        ),

        cf.embed_two_body(
            Sy, Sx,
            B4, C4,
            N_square,
            local_dim
        ),

        cf.embed_two_body(
            Sy, Sx,
            C4, D4,
            N_square,
            local_dim
        ),

        cf.embed_two_body(
            Sx, Sy,
            D4, A4,
            N_square,
            local_dim
        )
    ]


    active_channel = (
        1j
        * ordered_cycle_word(
            active_edges
        )
    )


    active_weights = cf.weight_decomposition(
        active_channel,
        local_dim=local_dim,
        n_sites=N_square
    )


    # Exact operator direction:
    #
    # K_active = s^{-3}
    #            Sz_A Sz_B Sz_C (Sx_D)^2

    active_target = (
        1 / spin**3
        * cf.kron_all([
            Sz,
            Sz,
            Sz,
            Sx_squared
        ])
    )


    active_direction_error = (
        np.linalg.norm(
            active_channel
            -
            active_target
        )
        /
        max(
            np.linalg.norm(
                active_target
            ),
            1e-300
        )
    )


    # --------------------------------------------------------------
    # All-conflict square:
    #
    # AB = Sx_A Sx_B
    # BC = Sy_B Sx_C
    # CD = Sy_C Sx_D
    # DA = Sy_D Sy_A
    # --------------------------------------------------------------

    null_edges = [

        active_edges[0],
        active_edges[1],
        active_edges[2],

        cf.embed_two_body(
            Sy, Sy,
            D4, A4,
            N_square,
            local_dim
        )
    ]


    conflict_channel = (
        1j
        * ordered_cycle_word(
            null_edges
        )
    )


    conflict_weights = cf.weight_decomposition(
        conflict_channel,
        local_dim=local_dim,
        n_sites=N_square
    )


    # Exact operator direction:
    #
    # K_conflict = (2 s^3)^{-1} [
    #     Sz_A Sz_B Sz_C {Sy,Sx}_D
    #     +
    #     {Sy,Sx}_A Sz_B Sz_C Sz_D
    # ]

    conflict_target = (

        1 / (2 * spin**3)

        * (

            cf.kron_all([
                Sz,
                Sz,
                Sz,
                Sxy_anticommutator
            ])

            +

            cf.kron_all([
                Sxy_anticommutator,
                Sz,
                Sz,
                Sz
            ])
        )
    )


    conflict_direction_error = (
        np.linalg.norm(
            conflict_channel
            -
            conflict_target
        )
        /
        max(
            np.linalg.norm(
                conflict_target
            ),
            1e-300
        )
    )


    spin_square_rows.append({

        "local_dim":
            local_dim,

        "spin":
            spin,

        "||traceless(Sx^2)||":
            np.linalg.norm(
                Sx_squared_traceless
            ),

        "||{Sx,Sy}||":
            np.linalg.norm(
                Sxy_anticommutator
            ),

        "active_channel_norm":
            np.linalg.norm(
                active_channel
            ),

        "active_weight3":
            active_weights[3],

        "active_weight4":
            active_weights[4],

        "active_direction_error":
            active_direction_error,

        "conflict_channel_norm":
            np.linalg.norm(
                conflict_channel
            ),

        "conflict_weight3":
            conflict_weights[3],

        "conflict_weight4":
            conflict_weights[4],

        "conflict_direction_error":
            conflict_direction_error
    })


spin_square_results = pd.DataFrame(
    spin_square_rows
)


display(
    spin_square_results
)

,local_dim,spin,||traceless(Sx^2)||,"||{Sx,Sy}||",active_channel_norm,active_weight3,active_weight4,active_direction_error,conflict_channel_norm,conflict_weight3,conflict_weight4,conflict_direction_error
0,2,0.5,0.000000,0.000000,32.000000,1.000000,0.000000,0.000000e+00,0.000000,0.000000e+00,0.0,0.000000e+00
1,3,1.0,0.816497,1.414214,4.000000,0.666667,0.333333,6.661338e-16,2.828427,0.000000e+00,1.0,6.661338e-16
2,4,1.5,0.888889,1.539601,1.396648,0.609756,0.390244,5.444386e-16,1.068564,2.867087e-33,1.0,5.128059e-16


In [49]:
spin_square_summary = spin_square_results[
    [
        "local_dim",
        "spin",
        "active_weight3",
        "active_weight4",
        "conflict_channel_norm",
        "conflict_weight4"
    ]
].copy()


display(
    spin_square_summary
)

,local_dim,spin,active_weight3,active_weight4,conflict_channel_norm,conflict_weight4
0,2,0.5,1.000000,0.000000,0.000000,0.0
1,3,1.0,0.666667,0.333333,2.828427,1.0
2,4,1.5,0.609756,0.390244,1.068564,1.0


In [50]:
# ------------------------------------------------------------------
# Acceptance tests
# ------------------------------------------------------------------

qubit_row = (
    spin_square_results
    .loc[
        spin_square_results[
            "local_dim"
        ] == 2
    ]
    .iloc[0]
)


higher_spin_rows = (
    spin_square_results
    .loc[
        spin_square_results[
            "local_dim"
        ] > 2
    ]
)


# --------------------------------------------------------------
# Spin 1/2:
# Sx^2 is scalar and {Sx,Sy}=0.
# --------------------------------------------------------------

assert (
    qubit_row[
        "||traceless(Sx^2)||"
    ]
    < 1e-12
)


assert (
    qubit_row[
        "||{Sx,Sy}||"
    ]
    < 1e-12
)


assert (
    qubit_row[
        "active_weight3"
    ]
    > 1 - 1e-12
)


assert (
    qubit_row[
        "active_weight4"
    ]
    < 1e-12
)


assert (
    qubit_row[
        "conflict_channel_norm"
    ]
    < 1e-12
)


# --------------------------------------------------------------
# Higher spin:
# quadrupolar local operators lift both qubit simplifications.
# --------------------------------------------------------------

assert (
    higher_spin_rows[
        "||traceless(Sx^2)||"
    ].min()
    > 1e-8
)


assert (
    higher_spin_rows[
        "||{Sx,Sy}||"
    ].min()
    > 1e-8
)


# Active square contains both the scalar part of Sx^2
# and its traceless quadrupolar part.

assert (
    higher_spin_rows[
        "active_weight3"
    ].min()
    > 1e-6
)


assert (
    higher_spin_rows[
        "active_weight4"
    ].min()
    > 1e-6
)


assert (
    (
        higher_spin_rows[
            "active_weight3"
        ]

        +

        higher_spin_rows[
            "active_weight4"
        ]

        - 1.0
    )
    .abs()
    .max()
    < 1e-10
)


# The former all-conflict null becomes a pure weight-4 channel.

assert (
    higher_spin_rows[
        "conflict_channel_norm"
    ].min()
    > 1e-8
)


assert (
    higher_spin_rows[
        "conflict_weight4"
    ].min()
    > 1 - 1e-10
)


assert (
    higher_spin_rows[
        "conflict_weight3"
    ].max()
    < 1e-10
)


# Direct numerical words agree with the analytic local-algebra forms.

assert (
    spin_square_results[
        "active_direction_error"
    ].max()
    < 1e-10
)


# The qubit conflict target is exactly zero, so its relative error
# is numerically undefined; test only the nonzero higher-spin rows.

assert (
    higher_spin_rows[
        "conflict_direction_error"
    ].max()
    < 1e-10
)


print(
    "Spin-1/2 active square remains pure weight-3: PASS"
)

print(
    "Spin-1/2 all-conflict square remains null: PASS"
)

print(
    "Higher-spin active square splits into weight-3 and weight-4: PASS"
)

print(
    "Higher-spin all-conflict null is lifted to weight-4: PASS"
)

print(
    "Analytic spin-s operator identities: PASS"
)


spin_square_results.to_csv(
    result_dir
    / "spin_s_square_cycle_tax.csv",
    index=False
)

Spin-1/2 active square remains pure weight-3: PASS
Spin-1/2 all-conflict square remains null: PASS
Higher-spin active square splits into weight-3 and weight-4: PASS
Higher-spin all-conflict null is lifted to weight-4: PASS
Analytic spin-s operator identities: PASS


Strang palindrome clean phantom-edge ledger

In [51]:
# ------------------------------------------------------------------
# Strang palindrome:
# clean pairwise phantom-edge benchmark
# ------------------------------------------------------------------

Cmt = cf.commutator


K_strang_cubic = (

    1 / 24
    * Cmt(
        H_AB,
        Cmt(
            H_AB,
            H_BC
        )
    )

    +

    1 / 12
    * Cmt(
        H_BC,
        Cmt(
            H_AB,
            H_BC
        )
    )
)

In [52]:
# ------------------------------------------------------------------
# Analytic pair ledger
# ------------------------------------------------------------------

strang_pair_sites = {
    "AB": {A, B},
    "BC": {B, C},
    "AC": {A, C}
}


K_strang_pair_targets = {

    pair_name:
        cf.exact_support_component(
            K_strang_cubic,
            sites,
            local_dim=2,
            n_sites=N
        )

    for pair_name, sites
    in strang_pair_sites.items()
}


K_strang_pair_sum = sum(
    K_strang_pair_targets.values(),
    np.zeros_like(
        K_strang_cubic
    )
)


strang_target_closure_error = (

    np.linalg.norm(
        K_strang_cubic
        -
        K_strang_pair_sum
    )

    /

    np.linalg.norm(
        K_strang_cubic
    )
)


strang_analytic_ledger = pd.DataFrame([

    {
        "pair":
            pair_name,

        "microscopic_edge":
            pair_name in {
                "AB",
                "BC"
            },

        "phantom_edge":
            pair_name == "AC",

        "analytic_component_norm":
            np.linalg.norm(
                component
            ),

        "squared_norm_fraction":
            np.linalg.norm(
                component
            ) ** 2

            /

            np.linalg.norm(
                K_strang_cubic
            ) ** 2
    }

    for pair_name, component
    in K_strang_pair_targets.items()
])


display(
    strang_analytic_ledger
)


print(
    "Analytic pair-ledger closure error:",
    f"{strang_target_closure_error:.3e}"
)

,pair,microscopic_edge,phantom_edge,analytic_component_norm,squared_norm_fraction
0,AB,True,False,4.483565,0.464878
1,BC,True,False,2.219392,0.113909
2,AC,False,True,4.267811,0.421213


Analytic pair-ledger closure error: 7.694e-17


In [53]:
# ------------------------------------------------------------------
# Finite-step Strang scaling
# ------------------------------------------------------------------

strang_epsilon_values = np.array([
    0.20,
    0.16,
    0.12,
    0.10,
    0.08,
    0.06,
    0.04,
    0.03,
    0.02
])


strang_rows = []


for epsilon in strang_epsilon_values:

    hams, areas = cf.strang_lists(
        H_AB,
        H_BC,
        epsilon
    )


    U_strang = cf.protocol_unitary(
        hams,
        areas
    )


    branch_distance, maximum_phase = (
        cf.unitary_branch_distance(
            U_strang
        )
    )


    L_strang = cf.log_unitary(
        U_strang
    )


    G_strang = (
        1j * L_strang
        / epsilon
    )


    delta_G_strang = (
        G_strang
        -
        H_open_micro
    )


    weight_fractions = cf.weight_decomposition(
        delta_G_strang,
        local_dim=2,
        n_sites=N
    )


    row = {

        "epsilon":
            epsilon,

        "correction_norm":
            np.linalg.norm(
                delta_G_strang
            ),

        "weight2_fraction":
            weight_fractions[2],

        "weight3_fraction":
            weight_fractions[3],

        "branch_distance":
            branch_distance,

        "maximum_abs_eigenphase":
            maximum_phase
    }


    for pair_name, sites in strang_pair_sites.items():

        observed_component = (
            cf.exact_support_component(
                delta_G_strang,
                sites,
                local_dim=2,
                n_sites=N
            )
        )


        target_component = (
            K_strang_pair_targets[
                pair_name
            ]
        )


        target_norm_squared = np.vdot(
            target_component,
            target_component
        ).real


        fitted_amplitude = (

            np.vdot(
                target_component,
                observed_component
            ).real

            /

            max(
                target_norm_squared,
                1e-300
            )
        )


        fitted_component = (
            fitted_amplitude
            * target_component
        )


        observed_norm = np.linalg.norm(
            observed_component
        )


        direction_residual = (

            np.linalg.norm(
                observed_component
                -
                fitted_component
            )

            /

            max(
                observed_norm,
                1e-300
            )
        )


        scaled_analytic_error = (

            np.linalg.norm(
                observed_component
                / epsilon**2
                -
                target_component
            )

            /

            np.linalg.norm(
                target_component
            )
        )


        row[
            f"{pair_name}_norm"
        ] = observed_norm


        row[
            f"{pair_name}_amplitude_over_epsilon2"
        ] = (
            fitted_amplitude
            / epsilon**2
        )


        row[
            f"{pair_name}_direction_residual"
        ] = direction_residual


        row[
            f"{pair_name}_scaled_analytic_error"
        ] = scaled_analytic_error


    strang_rows.append(
        row
    )


strang_phantom_results = pd.DataFrame(
    strang_rows
)


display(
    strang_phantom_results
)

,epsilon,correction_norm,weight2_fraction,weight3_fraction,branch_distance,maximum_abs_eigenphase,AB_norm,AB_amplitude_over_epsilon2,AB_direction_residual,AB_scaled_analytic_error,BC_norm,BC_amplitude_over_epsilon2,BC_direction_residual,BC_scaled_analytic_error,AC_norm,AC_amplitude_over_epsilon2,AC_direction_residual,AC_scaled_analytic_error
0,0.20,0.270858,1.0,2.612128e-28,2.455600,0.685993,0.186164,1.038020,0.005390,0.038430,0.085962,0.968171,0.016332,0.035541,0.176968,1.036609,0.007936,0.037522
1,0.16,0.171538,1.0,6.089749e-28,2.590604,0.550988,0.117556,1.024185,0.003398,0.024434,0.055710,0.980483,0.010115,0.021892,0.111813,1.023391,0.005018,0.023948
2,0.12,0.095701,1.0,5.201552e-27,2.727095,0.414497,0.065437,1.013537,0.001889,0.013672,0.031620,0.989385,0.005550,0.011951,0.062264,1.013136,0.002796,0.013438
3,0.10,0.066245,1.0,4.318477e-27,2.795772,0.345821,0.045256,1.009382,0.001306,0.009474,0.022033,0.992726,0.003818,0.008202,0.043067,1.009116,0.001934,0.009322
4,0.08,0.042285,1.0,3.359862e-26,2.864671,0.276921,0.028867,1.005995,0.000833,0.006053,0.014139,0.995395,0.002424,0.005199,0.027473,1.005831,0.001234,0.005961
5,0.06,0.023736,1.0,1.540105e-25,2.933748,0.207845,0.016195,1.003368,0.000467,0.003400,0.007969,0.997432,0.001356,0.002902,0.015414,1.003278,0.000693,0.003351
6,0.04,0.010534,1.0,1.214725e-24,3.002957,0.138636,0.007184,1.001495,0.000207,0.001510,0.003547,0.998865,0.000600,0.001283,0.006838,1.001456,0.000307,0.001489
7,0.03,0.005922,1.0,1.004580e-23,3.037597,0.103996,0.004039,1.000841,0.000116,0.000849,0.001996,0.999363,0.000337,0.000720,0.003844,1.000819,0.000173,0.000837
8,0.02,0.002631,1.0,9.010175e-23,3.072253,0.069340,0.001794,1.000374,0.000052,0.000377,0.000888,0.999717,0.000150,0.000320,0.001708,1.000364,0.000077,0.000372


In [54]:
# ------------------------------------------------------------------
# Scaling exponents
# ------------------------------------------------------------------

strang_fit_mask = (
    strang_phantom_results[
        "epsilon"
    ]
    <= 0.08
)


strang_pair_exponents = {}


for pair_name in [
    "AB",
    "BC",
    "AC"
]:

    exponent = np.polyfit(

        np.log(
            strang_phantom_results.loc[
                strang_fit_mask,
                "epsilon"
            ]
        ),

        np.log(
            strang_phantom_results.loc[
                strang_fit_mask,
                f"{pair_name}_norm"
            ]
        ),

        deg=1
    )[0]


    strang_pair_exponents[
        pair_name
    ] = exponent

In [55]:
smallest_strang_row = (
    strang_phantom_results
    .sort_values(
        "epsilon"
    )
    .iloc[0]
)


strang_scaling_summary = pd.DataFrame({

    "quantity": [
        "AB exponent",
        "BC exponent",
        "AC phantom exponent",
        "minimum correction weight-2",
        "maximum correction weight-3",
        "minimum branch distance",
        "smallest-epsilon AB scaled error",
        "smallest-epsilon BC scaled error",
        "smallest-epsilon AC scaled error",
        "smallest-epsilon AC amplitude / epsilon^2"
    ],

    "value": [
        strang_pair_exponents["AB"],
        strang_pair_exponents["BC"],
        strang_pair_exponents["AC"],

        strang_phantom_results[
            "weight2_fraction"
        ].min(),

        strang_phantom_results[
            "weight3_fraction"
        ].max(),

        strang_phantom_results[
            "branch_distance"
        ].min(),

        smallest_strang_row[
            "AB_scaled_analytic_error"
        ],

        smallest_strang_row[
            "BC_scaled_analytic_error"
        ],

        smallest_strang_row[
            "AC_scaled_analytic_error"
        ],

        smallest_strang_row[
            "AC_amplitude_over_epsilon2"
        ]
    ]
})


display(
    strang_scaling_summary
)

,quantity,value
0,AB exponent,2.003917e+00
1,BC exponent,1.996978e+00
2,AC phantom exponent,2.003811e+00
3,minimum correction weight-2,1.000000e+00
4,maximum correction weight-3,9.010175e-23
5,minimum branch distance,2.455600e+00
6,smallest-epsilon AB scaled error,3.771932e-04
7,smallest-epsilon BC scaled error,3.197300e-04
8,smallest-epsilon AC scaled error,3.720354e-04
9,smallest-epsilon AC amplitude / epsilon^2,1.000364e+00


In [56]:
# ------------------------------------------------------------------
# Acceptance tests
# ------------------------------------------------------------------

assert (
    strang_target_closure_error
    < 1e-12
)


# All three pair sectors, including the microscopic gap AC,
# are active in the analytic Strang correction.

assert (
    strang_analytic_ledger[
        "analytic_component_norm"
    ].min()
    > 1e-8
)


# Full finite-step correction remains pairwise.

assert (
    strang_phantom_results[
        "weight2_fraction"
    ].min()
    > 1 - 1e-10
)


assert (
    strang_phantom_results[
        "weight3_fraction"
    ].max()
    < 1e-10
)


# Each pair correction scales as epsilon^2.

for exponent in strang_pair_exponents.values():

    assert (
        1.95
        <
        exponent
        <
        2.05
    )


# Principal logarithm remains branch-safe.

assert (
    strang_phantom_results[
        "branch_distance"
    ].min()
    > 1.0
)


# At the smallest epsilon, all pair components converge
# to their analytic cubic directions.

smallest_scaled_errors = np.array([

    smallest_strang_row[
        "AB_scaled_analytic_error"
    ],

    smallest_strang_row[
        "BC_scaled_analytic_error"
    ],

    smallest_strang_row[
        "AC_scaled_analytic_error"
    ]
])


assert (
    smallest_scaled_errors.max()
    < 5e-4
)


smallest_amplitude_ratios = np.array([

    smallest_strang_row[
        "AB_amplitude_over_epsilon2"
    ],

    smallest_strang_row[
        "BC_amplitude_over_epsilon2"
    ],

    smallest_strang_row[
        "AC_amplitude_over_epsilon2"
    ]
])


assert (
    np.max(
        np.abs(
            smallest_amplitude_ratios
            -
            1.0
        )
    )
    < 5e-4
)


# Direction remains close to the analytic pair directions
# over the entire tested epsilon range.

maximum_direction_residual = (
    strang_phantom_results[
        [
            "AB_direction_residual",
            "BC_direction_residual",
            "AC_direction_residual"
        ]
    ]
    .to_numpy()
    .max()
)


assert (
    maximum_direction_residual
    < 2e-2
)


print(
    "Strang analytic pair ledger: PASS"
)

print(
    "Strang full-order pairwise closure: PASS"
)

print(
    "Existing-edge renormalizations scale as epsilon^2: PASS"
)

print(
    "AC phantom edge scales as epsilon^2: PASS"
)

print(
    "All pair directions agree with BCH prediction: PASS"
)

print(
    "Principal-log branch safety: PASS"
)

print()

print(
    "AB exponent:",
    f"{strang_pair_exponents['AB']:.6f}"
)

print(
    "BC exponent:",
    f"{strang_pair_exponents['BC']:.6f}"
)

print(
    "AC phantom exponent:",
    f"{strang_pair_exponents['AC']:.6f}"
)


strang_analytic_ledger.to_csv(
    result_dir
    / "strang_analytic_pair_ledger.csv",
    index=False
)


strang_phantom_results.to_csv(
    result_dir
    / "strang_phantom_scaling.csv",
    index=False
)


strang_scaling_summary.to_csv(
    result_dir
    / "strang_phantom_summary.csv",
    index=False
)

Strang analytic pair ledger: PASS
Strang full-order pairwise closure: PASS
Existing-edge renormalizations scale as epsilon^2: PASS
AC phantom edge scales as epsilon^2: PASS
All pair directions agree with BCH prediction: PASS
Principal-log branch safety: PASS

AB exponent: 2.003917
BC exponent: 1.996978
AC phantom exponent: 2.003811


Random ensemble robustness

In [57]:
# ------------------------------------------------------------------
# Random-ensemble robustness of the Strang phantom edge
# ------------------------------------------------------------------

ensemble_seed = 20260807
ensemble_rng = np.random.default_rng(
    ensemble_seed
)

n_ensemble_samples = 300
ensemble_epsilon = 0.04

ensemble_J_AB = []
ensemble_J_BC = []
ensemble_rows = []


def random_normalized_coupling(
    rng
):
    """
    Generic real 3x3 bilinear coupling with unit Frobenius norm.
    """

    coupling = rng.normal(
        size=(3, 3)
    )

    return (
        coupling
        /
        np.linalg.norm(
            coupling
        )
    )


for sample_index in range(
    n_ensemble_samples
):

    # --------------------------------------------------------------
    # Random microscopic open wedge
    # --------------------------------------------------------------

    J_AB_sample = random_normalized_coupling(
        ensemble_rng
    )

    J_BC_sample = random_normalized_coupling(
        ensemble_rng
    )


    H_AB_sample = cf.general_edge_hamiltonian(
        A,
        B,
        J_AB_sample,
        N
    )

    H_BC_sample = cf.general_edge_hamiltonian(
        B,
        C,
        J_BC_sample,
        N
    )


    H_micro_sample = (
        H_AB_sample
        +
        H_BC_sample
    )


    ensemble_J_AB.append(
        J_AB_sample
    )

    ensemble_J_BC.append(
        J_BC_sample
    )


    # --------------------------------------------------------------
    # Confirm that AC is absent microscopically
    # --------------------------------------------------------------

    microscopic_AC = cf.exact_support_component(
        H_micro_sample,
        {A, C},
        local_dim=2,
        n_sites=N
    )


    microscopic_AC_relative_norm = (

        np.linalg.norm(
            microscopic_AC
        )

        /

        np.linalg.norm(
            H_micro_sample
        )
    )


    # --------------------------------------------------------------
    # Analytic Strang cubic correction
    # --------------------------------------------------------------

    commutator_sample = cf.commutator


    K_cubic_sample = (

        1 / 24
        * commutator_sample(
            H_AB_sample,
            commutator_sample(
                H_AB_sample,
                H_BC_sample
            )
        )

        +

        1 / 12
        * commutator_sample(
            H_BC_sample,
            commutator_sample(
                H_AB_sample,
                H_BC_sample
            )
        )
    )


    K_AB_sample = cf.exact_support_component(
        K_cubic_sample,
        {A, B},
        local_dim=2,
        n_sites=N
    )

    K_BC_sample = cf.exact_support_component(
        K_cubic_sample,
        {B, C},
        local_dim=2,
        n_sites=N
    )

    K_AC_sample = cf.exact_support_component(
        K_cubic_sample,
        {A, C},
        local_dim=2,
        n_sites=N
    )


    K_pair_sum_sample = (
        K_AB_sample
        +
        K_BC_sample
        +
        K_AC_sample
    )


    analytic_closure_error = (

        np.linalg.norm(
            K_cubic_sample
            -
            K_pair_sum_sample
        )

        /

        max(
            np.linalg.norm(
                K_cubic_sample
            ),
            1e-300
        )
    )


    phantom_AC_norm = np.linalg.norm(
        K_AC_sample
    )


    phantom_relative_norm = (

        phantom_AC_norm

        /

        max(
            np.linalg.norm(
                K_cubic_sample
            ),
            1e-300
        )
    )


    phantom_squared_fraction = (
        phantom_relative_norm**2
    )


    # --------------------------------------------------------------
    # Exact finite-step Strang effective generator
    # --------------------------------------------------------------

    sample_hams, sample_areas = cf.strang_lists(
        H_AB_sample,
        H_BC_sample,
        ensemble_epsilon
    )


    U_sample = cf.protocol_unitary(
        sample_hams,
        sample_areas
    )


    (
        branch_distance,
        maximum_abs_eigenphase
    ) = cf.unitary_branch_distance(
        U_sample
    )


    L_sample = cf.log_unitary(
        U_sample
    )


    G_eff_sample = (
        1j
        * L_sample
        / ensemble_epsilon
    )


    delta_G_sample = (
        G_eff_sample
        -
        H_micro_sample
    )


    finite_weights = cf.weight_decomposition(
        delta_G_sample,
        local_dim=2,
        n_sites=N
    )


    observed_AB = cf.exact_support_component(
        delta_G_sample,
        {A, B},
        local_dim=2,
        n_sites=N
    )

    observed_BC = cf.exact_support_component(
        delta_G_sample,
        {B, C},
        local_dim=2,
        n_sites=N
    )

    observed_AC = cf.exact_support_component(
        delta_G_sample,
        {A, C},
        local_dim=2,
        n_sites=N
    )


    observed_pair_sum = (
        observed_AB
        +
        observed_BC
        +
        observed_AC
    )


    finite_pair_closure_error = (

        np.linalg.norm(
            delta_G_sample
            -
            observed_pair_sum
        )

        /

        max(
            np.linalg.norm(
                delta_G_sample
            ),
            1e-300
        )
    )


    # --------------------------------------------------------------
    # AC direction and amplitude recovery
    # --------------------------------------------------------------

    analytic_AC_norm_squared = np.vdot(
        K_AC_sample,
        K_AC_sample
    ).real


    fitted_AC_amplitude = (

        np.vdot(
            K_AC_sample,
            observed_AC
        ).real

        /

        max(
            analytic_AC_norm_squared,
            1e-300
        )
    )


    amplitude_over_epsilon2 = (

        fitted_AC_amplitude

        /

        ensemble_epsilon**2
    )


    fitted_AC_component = (
        fitted_AC_amplitude
        *
        K_AC_sample
    )


    AC_direction_residual = (

        np.linalg.norm(
            observed_AC
            -
            fitted_AC_component
        )

        /

        max(
            np.linalg.norm(
                observed_AC
            ),
            1e-300
        )
    )


    AC_scaled_analytic_error = (

        np.linalg.norm(
            observed_AC
            / ensemble_epsilon**2
            -
            K_AC_sample
        )

        /

        max(
            phantom_AC_norm,
            1e-300
        )
    )


    ensemble_rows.append({

        "sample":
            sample_index,

        "microscopic_AC_relative_norm":
            microscopic_AC_relative_norm,

        "analytic_total_norm":
            np.linalg.norm(
                K_cubic_sample
            ),

        "analytic_AC_norm":
            phantom_AC_norm,

        "phantom_relative_norm":
            phantom_relative_norm,

        "phantom_squared_fraction":
            phantom_squared_fraction,

        "analytic_pair_closure_error":
            analytic_closure_error,

        "finite_weight2_fraction":
            finite_weights[2],

        "finite_weight3_fraction":
            finite_weights[3],

        "finite_pair_closure_error":
            finite_pair_closure_error,

        "AC_amplitude_over_epsilon2":
            amplitude_over_epsilon2,

        "AC_direction_residual":
            AC_direction_residual,

        "AC_scaled_analytic_error":
            AC_scaled_analytic_error,

        "branch_distance":
            branch_distance,

        "maximum_abs_eigenphase":
            maximum_abs_eigenphase
    })


strang_ensemble_results = pd.DataFrame(
    ensemble_rows
)


# ------------------------------------------------------------------
# Ensemble summaries
# ------------------------------------------------------------------

phantom_activation_threshold = 1e-3


phantom_activation_rate = np.mean(

    strang_ensemble_results[
        "phantom_relative_norm"
    ]

    >
    phantom_activation_threshold
)


ensemble_quantiles = (

    strang_ensemble_results[
        [
            "analytic_AC_norm",
            "phantom_relative_norm",
            "phantom_squared_fraction",
            "AC_amplitude_over_epsilon2",
            "AC_direction_residual",
            "AC_scaled_analytic_error"
        ]
    ]

    .quantile(
        [
            0.00,
            0.01,
            0.05,
            0.50,
            0.95,
            0.99,
            1.00
        ]
    )

    .T
)


display(
    ensemble_quantiles
)


strang_ensemble_summary = pd.DataFrame({

    "quantity": [
        "number of samples",
        "phantom activation rate",
        "median phantom squared fraction",
        "minimum phantom relative norm",
        "maximum microscopic AC relative norm",
        "maximum analytic closure error",
        "minimum finite weight-2 fraction",
        "maximum finite weight-3 fraction",
        "maximum finite pair-closure error",
        "maximum AC scaled analytic error",
        "maximum AC direction residual",
        "maximum |AC amplitude / epsilon^2 - 1|",
        "minimum branch distance"
    ],

    "value": [
        n_ensemble_samples,

        phantom_activation_rate,

        strang_ensemble_results[
            "phantom_squared_fraction"
        ].median(),

        strang_ensemble_results[
            "phantom_relative_norm"
        ].min(),

        strang_ensemble_results[
            "microscopic_AC_relative_norm"
        ].max(),

        strang_ensemble_results[
            "analytic_pair_closure_error"
        ].max(),

        strang_ensemble_results[
            "finite_weight2_fraction"
        ].min(),

        strang_ensemble_results[
            "finite_weight3_fraction"
        ].max(),

        strang_ensemble_results[
            "finite_pair_closure_error"
        ].max(),

        strang_ensemble_results[
            "AC_scaled_analytic_error"
        ].max(),

        strang_ensemble_results[
            "AC_direction_residual"
        ].max(),

        np.max(
            np.abs(
                strang_ensemble_results[
                    "AC_amplitude_over_epsilon2"
                ]
                -
                1.0
            )
        ),

        strang_ensemble_results[
            "branch_distance"
        ].min()
    ]
})


display(
    strang_ensemble_summary
)


# ------------------------------------------------------------------
# Acceptance tests
# ------------------------------------------------------------------

# AC is absent from every microscopic Hamiltonian.

assert (
    strang_ensemble_results[
        "microscopic_AC_relative_norm"
    ].max()
    < 1e-12
)


# The analytic cubic correction closes in the three pair sectors.

assert (
    strang_ensemble_results[
        "analytic_pair_closure_error"
    ].max()
    < 1e-12
)


# Phantom generation is generic in the continuous random ensemble.

assert (
    phantom_activation_rate
    > 0.99
)


assert (
    strang_ensemble_results[
        "phantom_squared_fraction"
    ].median()
    > 0.10
)


# The finite-step Strang correction remains pairwise.

assert (
    strang_ensemble_results[
        "finite_weight2_fraction"
    ].min()
    > 1 - 1e-10
)


assert (
    strang_ensemble_results[
        "finite_weight3_fraction"
    ].max()
    < 1e-10
)


assert (
    strang_ensemble_results[
        "finite_pair_closure_error"
    ].max()
    < 1e-10
)


# The finite-step AC edge agrees with the analytic BCH prediction.

assert (
    strang_ensemble_results[
        "AC_scaled_analytic_error"
    ].max()
    < 1e-3
)


assert (
    strang_ensemble_results[
        "AC_direction_residual"
    ].max()
    < 1e-3
)


assert (
    np.max(
        np.abs(
            strang_ensemble_results[
                "AC_amplitude_over_epsilon2"
            ]
            -
            1.0
        )
    )
    < 1e-3
)


# Every sample remains branch-safe.

assert (
    strang_ensemble_results[
        "branch_distance"
    ].min()
    > 1.0
)


print(
    "Microscopic AC absent in every sample: PASS"
)

print(
    "Generic phantom-edge activation: PASS"
)

print(
    "Random-ensemble pairwise closure: PASS"
)

print(
    "Random-ensemble BCH direction recovery: PASS"
)

print(
    "Random-ensemble branch safety: PASS"
)

print()

print(
    "Phantom activation rate:",
    f"{phantom_activation_rate:.3f}"
)

print(
    "Median phantom squared fraction:",
    f"{strang_ensemble_results['phantom_squared_fraction'].median():.3f}"
)


strang_ensemble_results.to_csv(
    result_dir
    / "strang_random_ensemble.csv",
    index=False
)


strang_ensemble_summary.to_csv(
    result_dir
    / "strang_random_ensemble_summary.csv",
    index=False
)


np.savez(
    result_dir
    / "strang_random_ensemble_couplings.npz",

    J_AB=np.asarray(
        ensemble_J_AB
    ),

    J_BC=np.asarray(
        ensemble_J_BC
    ),

    seed=ensemble_seed,

    epsilon=ensemble_epsilon
)

,0.00,0.01,0.05,0.50,0.95,0.99,1.00
analytic_AC_norm,0.106135,0.160084,0.245990,0.504489,0.741414,0.782899,0.839371
phantom_relative_norm,0.188451,0.259623,0.340763,0.542851,0.661312,0.694000,0.739574
phantom_squared_fraction,0.035514,0.067404,0.116123,0.294687,0.437334,0.481637,0.546969
AC_amplitude_over_epsilon2,1.000130,1.000226,1.000349,1.000470,1.000565,1.000600,1.000645
AC_direction_residual,0.000026,0.000034,0.000057,0.000147,0.000291,0.000344,0.000383
AC_scaled_analytic_error,0.000297,0.000382,0.000412,0.000495,0.000581,0.000622,0.000659


,quantity,value
0,number of samples,3.000000e+02
1,phantom activation rate,1.000000e+00
2,median phantom squared fraction,2.946870e-01
3,minimum phantom relative norm,1.884511e-01
4,maximum microscopic AC relative norm,0.000000e+00
5,maximum analytic closure error,1.496194e-16
6,minimum finite weight-2 fraction,1.000000e+00
7,maximum finite weight-3 fraction,4.729774e-22
8,maximum finite pair-closure error,2.686338e-11
9,maximum AC scaled analytic error,6.592707e-04


Microscopic AC absent in every sample: PASS
Generic phantom-edge activation: PASS
Random-ensemble pairwise closure: PASS
Random-ensemble BCH direction recovery: PASS
Random-ensemble branch safety: PASS

Phantom activation rate: 1.000
Median phantom squared fraction: 0.295


Open-wedge algebraic null

In [58]:
# ------------------------------------------------------------------
# Open-wedge controls:
#
# 1. generic bilinear wedge          -> AC phantom active
# 2. commuting Pauli wedge           -> no correction
# 3. noncommuting Pauli-word wedge   -> correction active, AC null
# ------------------------------------------------------------------

control_epsilon_values = np.array([
    0.12,
    0.08,
    0.06,
    0.04,
    0.03,
    0.02
])


# ------------------------------------------------------------------
# Control Hamiltonians
# ------------------------------------------------------------------

# Commuting at the shared site B:
#
#   X_A X_B
#   X_B Y_C

H_AB_commuting = cf.two_body(
    cf.X,
    cf.X,
    A,
    B,
    N
)

H_BC_commuting = cf.two_body(
    cf.X,
    cf.Y,
    B,
    C,
    N
)


# Noncommuting at B, but each edge contains only one Pauli word:
#
#   X_A X_B
#   Y_B Y_C
#
# The first commutator is nonzero and weight-3, but the odd nested
# words close back onto the original AB and BC generators.

H_AB_null = cf.two_body(
    cf.X,
    cf.X,
    A,
    B,
    N
)

H_BC_null = cf.two_body(
    cf.Y,
    cf.Y,
    B,
    C,
    N
)


control_cases = {

    "generic bilinear": (
        H_AB,
        H_BC
    ),

    "commuting control": (
        H_AB_commuting,
        H_BC_commuting
    ),

    "noncommuting phantom-null": (
        H_AB_null,
        H_BC_null
    )
}

In [59]:
def strang_cubic_coefficient(
    H_left,
    H_right
):
    """
    Hermitian epsilon^3 coefficient K^(3) in i log U_Strang:

        U = exp(-i eps H_left / 2)
            exp(-i eps H_right)
            exp(-i eps H_left / 2).
    """

    first_commutator = cf.commutator(
        H_left,
        H_right
    )

    return (

        1 / 24
        * cf.commutator(
            H_left,
            first_commutator
        )

        +

        1 / 12
        * cf.commutator(
            H_right,
            first_commutator
        )
    )


control_analytic_rows = []
control_analytic_targets = {}


for case_name, (
    H_left,
    H_right
) in control_cases.items():

    first_commutator = cf.commutator(
        H_left,
        H_right
    )


    K_cubic = strang_cubic_coefficient(
        H_left,
        H_right
    )


    K_AB_component = cf.exact_support_component(
        K_cubic,
        {A, B},
        local_dim=2,
        n_sites=N
    )

    K_BC_component = cf.exact_support_component(
        K_cubic,
        {B, C},
        local_dim=2,
        n_sites=N
    )

    K_AC_component = cf.exact_support_component(
        K_cubic,
        {A, C},
        local_dim=2,
        n_sites=N
    )


    control_analytic_targets[
        case_name
    ] = {

        "total":
            K_cubic,

        "AB":
            K_AB_component,

        "BC":
            K_BC_component,

        "AC":
            K_AC_component
    }


    total_norm = np.linalg.norm(
        K_cubic
    )


    control_analytic_rows.append({

        "case":
            case_name,

        "first_commutator_norm":
            np.linalg.norm(
                first_commutator
            ),

        "analytic_correction_norm":
            total_norm,

        "analytic_AB_norm":
            np.linalg.norm(
                K_AB_component
            ),

        "analytic_BC_norm":
            np.linalg.norm(
                K_BC_component
            ),

        "analytic_AC_norm":
            np.linalg.norm(
                K_AC_component
            ),

        "analytic_AC_fraction":
            np.linalg.norm(
                K_AC_component
            ) ** 2
            /
            max(
                total_norm**2,
                1e-300
            )
    })


control_analytic_results = pd.DataFrame(
    control_analytic_rows
)


display(
    control_analytic_results
)

,case,first_commutator_norm,analytic_correction_norm,analytic_AB_norm,analytic_BC_norm,analytic_AC_norm,analytic_AC_fraction
0,generic bilinear,16.120084,6.575885,4.483565,2.219392,4.267811,0.421213
1,commuting control,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
2,noncommuting phantom-null,5.656854,1.054093,0.942809,0.471405,0.000000,0.000000


In [60]:
control_finite_rows = []


for case_name, (
    H_left,
    H_right
) in control_cases.items():

    H_micro = (
        H_left
        +
        H_right
    )


    for epsilon in control_epsilon_values:

        hams, areas = cf.strang_lists(
            H_left,
            H_right,
            epsilon
        )


        U_control = cf.protocol_unitary(
            hams,
            areas
        )


        L_control = cf.log_unitary(
            U_control
        )


        G_control = (
            1j * L_control
            / epsilon
        )


        delta_G_control = (
            G_control
            -
            H_micro
        )


        delta_AB = cf.exact_support_component(
            delta_G_control,
            {A, B},
            local_dim=2,
            n_sites=N
        )

        delta_BC = cf.exact_support_component(
            delta_G_control,
            {B, C},
            local_dim=2,
            n_sites=N
        )

        delta_AC = cf.exact_support_component(
            delta_G_control,
            {A, C},
            local_dim=2,
            n_sites=N
        )


        correction_norm = np.linalg.norm(
            delta_G_control
        )


        branch_distance, _ = (
            cf.unitary_branch_distance(
                U_control
            )
        )


        control_finite_rows.append({

            "case":
                case_name,

            "epsilon":
                epsilon,

            "correction_norm":
                correction_norm,

            "AB_norm":
                np.linalg.norm(
                    delta_AB
                ),

            "BC_norm":
                np.linalg.norm(
                    delta_BC
                ),

            "AC_norm":
                np.linalg.norm(
                    delta_AC
                ),

            "AC_relative_norm":
                np.linalg.norm(
                    delta_AC
                )
                /
                max(
                    correction_norm,
                    1e-300
                ),

            "branch_distance":
                branch_distance
        })


control_finite_results = pd.DataFrame(
    control_finite_rows
)


display(
    control_finite_results
)

,case,epsilon,correction_norm,AB_norm,BC_norm,AC_norm,AC_relative_norm,branch_distance
0,generic bilinear,0.12,9.570122e-02,6.543745e-02,3.162047e-02,6.226402e-02,6.506084e-01,2.727095
1,generic bilinear,0.08,4.228450e-02,2.886684e-02,1.413874e-02,2.747327e-02,6.497243e-01,2.864671
2,generic bilinear,0.06,2.373606e-02,1.619519e-02,7.969297e-03,1.541449e-02,6.494123e-01,2.933748
3,generic bilinear,0.04,1.053383e-02,7.184432e-03,3.546999e-03,6.838443e-03,6.491887e-01,3.002957
4,generic bilinear,0.03,5.922224e-03,4.038602e-03,1.996181e-03,3.844176e-03,6.491103e-01,3.037597
5,generic bilinear,0.02,2.631130e-03,1.794096e-03,8.875058e-04,1.707746e-03,6.490542e-01,3.072253
6,commuting control,0.12,4.262504e-15,1.209527e-15,1.151869e-15,2.579419e-15,6.051417e-01,2.901593
7,commuting control,0.08,7.294167e-15,2.729358e-15,2.220019e-15,3.661683e-15,5.020015e-01,2.981593
8,commuting control,0.06,1.075029e-14,4.455259e-15,4.562435e-15,3.287027e-15,3.057617e-01,3.021593
9,commuting control,0.04,1.365505e-14,2.648358e-15,2.223009e-15,6.210703e-15,4.548282e-01,3.061593


In [61]:
control_summary_rows = []


for case_name in control_cases:

    analytic_row = (
        control_analytic_results
        .loc[
            control_analytic_results[
                "case"
            ] == case_name
        ]
        .iloc[0]
    )


    finite_rows = (
        control_finite_results
        .loc[
            control_finite_results[
                "case"
            ] == case_name
        ]
    )


    control_summary_rows.append({

        "case":
            case_name,

        "commutator_norm":
            analytic_row[
                "first_commutator_norm"
            ],

        "analytic_correction_norm":
            analytic_row[
                "analytic_correction_norm"
            ],

        "analytic_AC_norm":
            analytic_row[
                "analytic_AC_norm"
            ],

        "maximum_finite_AC_norm":
            finite_rows[
                "AC_norm"
            ].max(),

        "maximum_finite_AC_relative_norm":
            finite_rows[
                "AC_relative_norm"
            ].max(),

        "minimum_branch_distance":
            finite_rows[
                "branch_distance"
            ].min()
    })


control_summary = pd.DataFrame(
    control_summary_rows
)


display(
    control_summary
)

,case,commutator_norm,analytic_correction_norm,analytic_AC_norm,maximum_finite_AC_norm,maximum_finite_AC_relative_norm,minimum_branch_distance
0,generic bilinear,16.120084,6.575885,4.267811,6.226402e-02,6.506084e-01,2.727095
1,commuting control,0.000000,0.000000,0.000000,1.181789e-14,6.051417e-01,2.901593
2,noncommuting phantom-null,5.656854,1.054093,0.000000,1.266860e-14,3.004466e-11,2.972091


In [ ]:
# ------------------------------------------------------------------
# Acceptance tests
# ------------------------------------------------------------------

generic_row = (
    control_summary
    .loc[
        control_summary[
            "case"
        ] == "generic bilinear"
    ]
    .iloc[0]
)


commuting_row = (
    control_summary
    .loc[
        control_summary[
            "case"
        ] == "commuting control"
    ]
    .iloc[0]
)


null_row = (
    control_summary
    .loc[
        control_summary[
            "case"
        ] == "noncommuting phantom-null"
    ]
    .iloc[0]
)


# Generic sample:
# noncommuting and phantom-active.

assert (
    generic_row[
        "commutator_norm"
    ]
    > 1e-6
)


assert (
    generic_row[
        "analytic_AC_norm"
    ]
    > 1e-6
)


# Commuting control:
# no commutator and no Strang correction.

assert (
    commuting_row[
        "commutator_norm"
    ]
    < 1e-12
)


assert (
    commuting_row[
        "analytic_correction_norm"
    ]
    < 1e-12
)


assert (
    commuting_row[
        "maximum_finite_AC_norm"
    ]
    < 1e-12
)


# Algebraic null:
# first commutator is nonzero, cubic correction is nonzero,
# but AC projection vanishes.

assert (
    null_row[
        "commutator_norm"
    ]
    > 1e-6
)


assert (
    null_row[
        "analytic_correction_norm"
    ]
    > 1e-6
)


assert (
    null_row[
        "analytic_AC_norm"
    ]
    < 1e-12
)


assert (
    null_row[
        "maximum_finite_AC_norm"
    ]
    < 1e-10
)


# All cases remain branch-safe.

assert (
    control_summary[
        "minimum_branch_distance"
    ].min()
    > 1.0
)


print(
    "Generic bilinear wedge generates AC: PASS"
)

print(
    "Commuting wedge produces no correction: PASS"
)

print(
    "Noncommuting Pauli wedge has active BCH correction: PASS"
)

print(
    "Noncommuting Pauli wedge remains phantom-null: PASS"
)

print(
    "All control protocols are branch-safe: PASS"
)


control_analytic_results.to_csv(
    result_dir
    / "open_wedge_null_controls_analytic.csv",
    index=False
)


control_finite_results.to_csv(
    result_dir
    / "open_wedge_null_controls_finite.csv",
    index=False
)


control_summary.to_csv(
    result_dir
    / "open_wedge_null_controls_summary.csv",
    index=False
)

learner benchmark, for stage 3

In [63]:
# ------------------------------------------------------------------
# Freeze Stage 2 results and export the learner benchmark
# ------------------------------------------------------------------

stage2_claim_rows = [

    {
        "claim_id": "C1",
        "claim": (
            "Centered multidegree stencils recover the analytic "
            "BCH coefficients with Richardson acceleration."
        ),
        "diagnostic": (
            multidegree_anchor_results[
                "richardson_error"
            ].max()
        ),
        "acceptance_bound": "< 1e-6",
        "status": "PASS"
    },

    {
        "claim_id": "C2",
        "claim": (
            "The complete cubic generator reconstructs as "
            "six directed wedge sectors plus the trilinear cycle sector."
        ),
        "diagnostic": reconstruction_relative_error,
        "acceptance_bound": "< 2e-6",
        "status": "PASS"
    },

    {
        "claim_id": "C3",
        "claim": (
            "An open microscopic wedge AB--BC generates a nonzero "
            "effective AC interaction."
        ),
        "diagnostic": minimum_phantom_fraction,
        "acceptance_bound": "> 1e-6",
        "status": "PASS"
    },

    {
        "claim_id": "C4",
        "claim": (
            "The asymmetric finite-step AC phantom edge scales "
            "as epsilon squared."
        ),
        "diagnostic": phantom_scaling_slope,
        "acceptance_bound": "1.95 < exponent < 2.05",
        "status": "PASS"
    },

    {
        "claim_id": "C5",
        "claim": (
            "The nested group-commutator witness isolates a cubic "
            "Lie channel, while the Kitaev null begins at fourth order."
        ),
        "diagnostic": kitaev_null_slope,
        "acceptance_bound": "3.8 < exponent < 4.2",
        "status": "PASS"
    },

    {
        "claim_id": "C6",
        "claim": (
            "The asymmetric Trotter correction begins as pure "
            "weight three, whereas the Strang logarithm remains pairwise."
        ),
        "diagnostic": (
            protocol_comparison_results[
                "palindrome_full_weight3"
            ].max()
        ),
        "acceptance_bound": "< 1e-10",
        "status": "PASS"
    },

    {
        "claim_id": "C7",
        "claim": (
            "Cycle support is constrained by motif topology but selected "
            "by the local operator algebra."
        ),
        "diagnostic": (
            higher_spin_rows[
                "active_weight4"
            ].min()
        ),
        "acceptance_bound": "> 1e-6 for d > 2",
        "status": "PASS"
    },

    {
        "claim_id": "C8",
        "claim": (
            "The Strang protocol produces a clean, entirely pairwise "
            "AC phantom edge scaling as epsilon squared."
        ),
        "diagnostic": strang_pair_exponents["AC"],
        "acceptance_bound": "1.95 < exponent < 2.05",
        "status": "PASS"
    },

    {
        "claim_id": "C9",
        "claim": (
            "Phantom AC generation is generic in the continuous "
            "random bilinear ensemble."
        ),
        "diagnostic": phantom_activation_rate,
        "acceptance_bound": "> 0.99",
        "status": "PASS"
    },

    {
        "claim_id": "C10",
        "claim": (
            "Ordinary noncommutativity alone is insufficient: "
            "a noncommuting single-Pauli wedge can remain phantom-null."
        ),
        "diagnostic": null_row[
            "analytic_AC_norm"
        ],
        "acceptance_bound": "< 1e-12",
        "status": "PASS"
    }
]


stage2_claim_manifest = pd.DataFrame(
    stage2_claim_rows
)


display(
    stage2_claim_manifest
)

,claim_id,claim,diagnostic,acceptance_bound,status
0,C1,Centered multidegree stencils recover the anal...,1.840149e-08,< 1e-6,PASS
1,C2,The complete cubic generator reconstructs as s...,9.149670e-07,< 2e-6,PASS
2,C3,An open microscopic wedge AB--BC generates a n...,3.773504e-01,> 1e-6,PASS
3,C4,The asymmetric finite-step AC phantom edge sca...,2.002079e+00,1.95 < exponent < 2.05,PASS
4,C5,The nested group-commutator witness isolates a...,3.991410e+00,3.8 < exponent < 4.2,PASS
5,C6,The asymmetric Trotter correction begins as pu...,1.274929e-29,< 1e-10,PASS
6,C7,Cycle support is constrained by motif topology...,3.333333e-01,> 1e-6 for d > 2,PASS
7,C8,"The Strang protocol produces a clean, entirely...",2.003811e+00,1.95 < exponent < 2.05,PASS
8,C9,Phantom AC generation is generic in the contin...,1.000000e+00,> 0.99,PASS
9,C10,Ordinary noncommutativity alone is insufficien...,0.000000e+00,< 1e-12,PASS


In [64]:
# ------------------------------------------------------------------
# Stage 3 benchmark package
# ------------------------------------------------------------------

learner_benchmark_epsilon_values = np.array([
    0.20,
    0.16,
    0.12,
    0.10,
    0.08,
    0.06,
    0.04,
    0.03,
    0.02
])


# Microscopic topology:
#
#     A ---- B ---- C
#
# with no microscopic AC edge.

learner_benchmark = {

    "H_AB":
        H_AB,

    "H_BC":
        H_BC,

    "H_micro":
        H_open_micro,

    "K_strang_cubic":
        K_strang_cubic,

    "K_AB_renormalization":
        K_strang_pair_targets["AB"],

    "K_BC_renormalization":
        K_strang_pair_targets["BC"],

    "K_AC_phantom":
        K_strang_pair_targets["AC"]
}

In [65]:
# ------------------------------------------------------------------
# Final benchmark integrity checks
# ------------------------------------------------------------------

benchmark_H_AC_micro = cf.exact_support_component(
    learner_benchmark["H_micro"],
    {A, C},
    local_dim=2,
    n_sites=N
)


benchmark_K_AC_phantom = (
    learner_benchmark[
        "K_AC_phantom"
    ]
)


benchmark_pair_sum = (

    learner_benchmark[
        "K_AB_renormalization"
    ]

    +

    learner_benchmark[
        "K_BC_renormalization"
    ]

    +

    learner_benchmark[
        "K_AC_phantom"
    ]
)


benchmark_closure_error = (

    np.linalg.norm(
        learner_benchmark[
            "K_strang_cubic"
        ]
        -
        benchmark_pair_sum
    )

    /

    np.linalg.norm(
        learner_benchmark[
            "K_strang_cubic"
        ]
    )
)


assert (
    np.linalg.norm(
        benchmark_H_AC_micro
    )
    < 1e-12
)


assert (
    np.linalg.norm(
        benchmark_K_AC_phantom
    )
    > 1e-6
)


assert (
    benchmark_closure_error
    < 1e-12
)


assert (
    (
        stage2_claim_manifest[
            "status"
        ]
        == "PASS"
    ).all()
)

In [66]:
stage2_claim_manifest.to_csv(
    result_dir
    / "stage2_claim_manifest.csv",
    index=False
)


np.savez(
    result_dir
    / "stage3_learner_benchmark.npz",

    H_AB=learner_benchmark[
        "H_AB"
    ],

    H_BC=learner_benchmark[
        "H_BC"
    ],

    H_micro=learner_benchmark[
        "H_micro"
    ],

    K_strang_cubic=learner_benchmark[
        "K_strang_cubic"
    ],

    K_AB_renormalization=learner_benchmark[
        "K_AB_renormalization"
    ],

    K_BC_renormalization=learner_benchmark[
        "K_BC_renormalization"
    ],

    K_AC_phantom=learner_benchmark[
        "K_AC_phantom"
    ],

    J_AB=J_AB,

    J_BC=J_BC,

    epsilon_values=
        learner_benchmark_epsilon_values,

    convention_version=
        cf.CONVENTION_VERSION,

    reference_seed=20260805
)


print(
    "Stage 2 claim manifest: PASS"
)

print(
    "Stage 3 learner benchmark integrity: PASS"
)

print()

print(
    "Microscopic AC norm:",
    f"{np.linalg.norm(benchmark_H_AC_micro):.3e}"
)

print(
    "Analytic phantom AC norm:",
    f"{np.linalg.norm(benchmark_K_AC_phantom):.6f}"
)

print(
    "Benchmark pair-closure error:",
    f"{benchmark_closure_error:.3e}"
)

print()

print(
    "Saved learner benchmark to:"
)

print(
    (
        result_dir
        / "stage3_learner_benchmark.npz"
    ).resolve()
)

Stage 2 claim manifest: PASS
Stage 3 learner benchmark integrity: PASS

Microscopic AC norm: 0.000e+00
Analytic phantom AC norm: 4.267811
Benchmark pair-closure error: 7.694e-17

Saved learner benchmark to:
C:\Users\liu.xuanc\Desktop\Code\Quantum-HOC\results_stage2\stage3_learner_benchmark.npz
